**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/souza-2026"], check=True)
    os.chdir("/content/encore/entries/souza-2026")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Souza et al. (2026) on DANDI:000713

Souza, Vieira, Ventura, de Lima, Arcanjo, de Paula, Sales, Souza, Alvarez-Jácobo, Fontenele, Aguiar,
de Santana, Sousa and de Vasconcelos (2026), "Reward coding across mouse visual cortical areas",
*Front. Behav. Neurosci.* 20:1944695, doi:10.3389/fnbeh.2026.1944695.

**Dataset.** DANDI:000713, "Allen Institute - Visual Behavior - Neuropixels", version
`0.240702.1725` (doi:10.48324/dandi.000713/0.240702.1725). This is the version the paper cites
(Materials and methods 2.1), and the only published version; the draft has the same asset count and
size and was last modified before the version was published. Files are streamed from the archive;
nothing is downloaded.

**Text.** The claims file `claims/souza-2026.yaml` was written from the version of record (Brief
Research Report, 28 October 2026), its Methods, figure legends and the legend of Supplementary
Figure S1. The paper links no analysis code.

**Claims.** All 38 claims are tested. The five layer claims also use the CCFv3 annotation volume, an
outside input (section "Cortical layers"). This is the revision after the first verification round.

| Claim | Status |
|---|---|
| data-sessions-units | tested |
| blocks-no-reward-in-replay | tested |
| reward-interval-min | tested |
| latency-change-to-reward | tested |
| lick-reward-coincide | tested |
| fig1e-cohort-size | tested |
| fig1d-split-half-above-null | tested |
| fig1c-example-cells | tested |
| fig1e-suppressed-fraction-by-area | tested |
| fig1l-session-gradient | tested |
| latency-jitter-control | tested |
| waveform-ambiguous-band | tested |
| fig1jk-narrow-fraction-by-area | tested |
| fig1h-median-isi-narrow-broad | tested |
| fig1gi-waveform-separation | tested |
| narrow-more-suppressed | tested |
| narrow-more-suppressed-within-layer | tested (outside input) |
| fig1mn-modulation-index-negative | tested |
| fig1m-narrow-stronger-m | tested |
| m-baseline-sensitivity | tested |
| layer-assignment-coverage | tested (outside input) |
| layer4-suppression | tested (outside input) |
| layer4-within-waveform | tested (outside input) |
| layer4-controls | tested (outside input) |
| fig2c-cv-lower | tested |
| cohort-faster-firing | tested |
| cv-rate-matched | tested |
| cv-unmatched-reverses | tested |
| fig2d-cv-correlations | tested |
| cv-correlation-block-excess | tested |
| fig2f-pre-post-cv | tested |
| fig3a-archetypes | tested |
| fig3b-mixtures | tested |
| fig3c-area-overlap | tested |
| fig3d-anticipatory-gradient | tested |
| anticipatory-direction-overlap | tested |
| hits-vs-false-alarms | tested |
| hits-vs-lick-bouts | tested |

Random seeds: each claim's generator is seeded with `zlib.crc32(b"souza-2026/<claim id>")`; where a
claim needs one generator per session, session k adds k.


In [ ]:
import os
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[v] = "1"
import time, zlib, warnings, resource
import numpy as np, pandas as pd
from scipy import stats
import vbn
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 200)
T0 = time.time()
KEY = "souza-2026"
def seed(claim):
    return zlib.crc32(f"{KEY}/{claim}".encode())
A = vbn.AREAS
print("dandiset", vbn.DANDISET, "version", vbn.VERSION)

## Data check

Every session file of the pinned version is listed, and the units table of each is read for its
area and quality columns. Then the sessions this attempt analyses are streamed and reduced
(claim `data-sessions-units` decides which).

In [ ]:
ses, assets = vbn.list_sessions()
print("assets in the version:", len(assets), "| session files:", len(ses), "| total size of session files %.0f GB" % (ses["size"].sum() / 1e9))
import concurrent.futures as cf
with cf.ThreadPoolExecutor(16) as ex:
    light = list(ex.map(vbn.read_light, ses.asset_id))
rows = []
for s, L in zip(ses.itertuples(), light):
    u = L["units"]; v = u[u.location.isin(A)]
    qc = (v.amplitude_cutoff <= 0.1) & (v.presence_ratio >= 0.95) & (v.isi_violations <= 0.5)
    rows.append(dict(path=s.path, asset_id=s.asset_id, subject=L["subject"], line=L["genotype"].split("/")[0],
                     session_type=L["session_type"], date=L["session_start_time"], n_units=len(u), n_vis=len(v), n_vis_allen_qc=int(qc.sum()),
                     n_vis_good=int((v.quality == "good").sum()),
                     n_vis_good_pr95=int(((v.quality == "good") & (v.presence_ratio >= 0.95)).sum()),
                     n_areas=v.location.nunique()))
S0 = pd.DataFrame(rows)
print(S0.groupby("line").agg(sessions=("path", "size"), mice=("subject", "nunique"), vis_units=("n_vis", "sum"),
      allen_qc=("n_vis_allen_qc", "sum"), good=("n_vis_good", "sum"), good_pr95=("n_vis_good_pr95", "sum")))
print(S0.session_type.value_counts())
print("sessions with all six areas:", (S0.n_areas == 6).sum())
print("elapsed %.0f s" % (time.time() - T0))

### Claim data-sessions-units

> The dandiset holds the data the paper analyzed, 103 sessions from 54 mice with 54,039 units in the
> six visual areas, with an active behavior block 0 and a passive replay block 5 in every session.

Readings of the criterion: (a) every session with blocks 0 and 5 and performance rewards, every unit
the units table assigns to the six areas; (b) the same sessions, units passing the Allen default
quality thresholds (amplitude cutoff <= 0.1, presence ratio >= 0.95, ISI violations <= 0.5). Other
selections the files allow are shown after them; they are not readings the criterion lists.

In [ ]:
def counts(mask, col):
    d = S0[mask]
    return len(d), d.subject.nunique(), int(d[col].sum())
allm = np.ones(len(S0), bool)
opto = (S0.line != "wt").values
tab = pd.DataFrame([
    ("(a) all sessions, all units of the six areas",) + counts(allm, "n_vis"),
    ("(b) all sessions, Allen default quality thresholds",) + counts(allm, "n_vis_allen_qc"),
    ("all sessions, quality 'good'",) + counts(allm, "n_vis_good"),
    ("all sessions, quality 'good' and presence ratio >= 0.95",) + counts(allm, "n_vis_good_pr95"),
    ("Sst-Cre and Vip-Cre mice, all units",) + counts(opto, "n_vis"),
    ("Sst-Cre and Vip-Cre mice, Allen default thresholds",) + counts(opto, "n_vis_allen_qc"),
    ("Sst-Cre and Vip-Cre mice, quality 'good'",) + counts(opto, "n_vis_good"),
    ("Sst-Cre and Vip-Cre mice, quality 'good' and presence ratio >= 0.95",) + counts(opto, "n_vis_good_pr95"),
], columns=["selection", "sessions", "mice", "units"])
tab["sessions vs 103"] = (tab.sessions / 103 - 1).round(3)
tab["mice vs 54"] = (tab.mice / 54 - 1).round(3)
tab["units vs 54039"] = (tab.units / 54039 - 1).round(3)
print(tab.to_string())

**Result: partially reproduced, resting on an assumption.** Neither reading the criterion lists matches: all 153 sessions from 81 mice hold 110,563 units in the six areas, or 45,354 with the Allen default thresholds. The sessions of the 54 Sst-Cre and Vip-Cre mice (104 sessions) with units of quality "good" and presence ratio >= 0.95 give 104 sessions, 54 mice and 54,045 units, within 10% of 103, 54 and 54,039 but not exact. This selection is used by every other claim, as an assumption. No session stands out as the one the paper left out: the smallest holds 262 units, so leaving one out cannot give 54,039 units from 54,045, and the session with the fewest hits (3) holds 903 units. Every one of these sessions has an active block 0 and a passive replay block 5 (claim blocks-no-reward-in-replay).

### Streaming the analysed sessions

The sessions of the Sst-Cre and Vip-Cre mice, units with quality "good" and presence ratio >= 0.95
in the six areas (the selection above that matches the paper's counts; an assumption). Each session
is reduced by `vbn.process_session`: reward-aligned histograms, split-half reliability and its
ISI-shuffle null (100 permutations; the paper uses 1,000, see the deviations), per-trial amplitudes and
the family tests, whole-recording spike counts, ISIs, 50 ms population counts of the active block,
counts around each image change, and the peak-channel mean waveform measures.

In [ ]:
SEL = S0[opto].reset_index(drop=True)
NPERM = 100
t = time.time()
OUT = vbn.run_sessions(SEL.asset_id.tolist(), NPERM, seed("fig1e-cohort-size"), seed("latency-jitter-control"), workers=6)
print("sessions processed:", len(OUT), "in %.0f s" % (time.time() - t))
U = pd.concat([o[1] for o in OUT], ignore_index=True)
U["subject"] = SEL.subject.values[U.session.values]
EV = [o[3] for o in OUT]
META = [o[0] for o in OUT]
print("units:", len(U), "| peak memory (main) %.1f GB" % (resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6))

In [ ]:
print("units by area:"); print(U.location.value_counts().reindex(A))
print("sessions per mouse:", SEL.groupby("subject").size().value_counts().to_dict())
print("session types:", SEL.session_type.value_counts().to_dict())
print("units per session: min %d, median %d, max %d" % tuple(U.groupby("session").size().agg(["min", "median", "max"])))
d = pd.DataFrame([dict(session=k, active_start=e["a0"], active_stop=e["a1"], active_min=(e["a1"] - e["a0"]) / 60,
                       replay_start=e["p0"], replay_stop=e["p1"], replay_min=(e["p1"] - e["p0"]) / 60,
                       rewards=len(e["rew_all"]), auto=int(e["rew_auto"].sum()), earned=len(e["earned"]),
                       earned_in_window=len(e["rew"]), hits=e["n_hit_trials"], misses=len(e["miss_change"]),
                       fa=e["n_fa_trials"], replay_changes=len(e["pas_changes"]), licks=len(e["licks"]),
                       med_latency=e["med_lat"]) for k, e in enumerate(EV)])
print(d.describe().T[["min", "50%", "max"]])
print("spike times (s): first %.1f, last %.1f" % (U.t_first.min(), U.t_last.max()))
print("waveform duration from the mean waveform (ms): %.3f to %.3f, finite for %d units; stored waveform_duration %.3f to %.3f"
      % (U.wf_dur.min(), U.wf_dur.max(), np.isfinite(U.wf_dur).sum(), U.waveform_duration.min(), U.waveform_duration.max()))
print("half-width (ms): %.3f to %.3f" % (U.wf_hw.min(), U.wf_hw.max()))
print("stored firing_rate (Hz): %.3f to %.2f" % (U.firing_rate.min(), U.firing_rate.max()))
print("CCF x, y, z (um): ", [(round(U[c].min()), round(U[c].max())) for c in "xyz"])
DUR = d
# per-session table of the analysed sessions, to look for the one the paper may have left out (verifier round 1)
SES = pd.DataFrame([dict(session=k, subject=SEL.subject[k], date=SEL.date[k], line=SEL.line[k], session_type=SEL.session_type[k],
                         units=int((U.session == k).sum()), areas=U[U.session == k].location.nunique(), hits=e["n_hit_trials"],
                         earned=len(e["earned"]), fa=e["n_fa_trials"]) for k, e in enumerate(EV)])
one = SES.groupby("subject").session.transform("size") == 1
print("sessions sorted by units (fewest 8):"); print(SES.sort_values("units").head(8).to_string(index=False))
print("sessions with fewer than six areas:", SES[SES.areas < 6][["session", "subject", "date", "areas", "units"]].to_dict("records"))
print("sessions with the fewest hits:"); print(SES.sort_values("hits").head(5).to_string(index=False))
print("mice with one session:", SES[one][["session", "subject", "date", "line", "units"]].to_dict("records"))
print("dates: %s to %s; sessions sharing a date with another of the same mouse: %d" % (SES.date.min(), SES.date.max(), SES.duplicated(["subject", "date"], keep=False).sum()))
print("smallest session has %d units, so leaving out one session cannot give 54,039 from %d" % (SES.units.min(), len(U)))


**Clock check.** The trial table, rewards, licks and spikes share the session clock if the units'
response to image changes is locked to the change times in the stimulus table. Below, the summed
population count of each session around image changes of the active block is compared between the 0.3 s after and before the change, and the same ratio is computed with the
change times shifted 0.25 s earlier; a decoder of post-change against pre-change windows is run on single-event counts, with the
change times shifted 0.25 s earlier as the reference.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
clock = []
for k, (meta, df, arr, ev) in enumerate(OUT):
    pp = arr["pp_act"].astype(float).sum(0)  # (changes, 30) summed over units, 50 ms bins from -0.75 s
    pre = pp[:, 5:15].mean(1); post = pp[:, 15:21].mean(1)  # -0.5..0 and 0..0.3 s
    # decoding: post-change window against the window just before the change
    X = np.r_[pp[:, 15:21], pp[:, 9:15]]; y = np.r_[np.ones(len(pp)), np.zeros(len(pp))]
    acc = cross_val_score(LogisticRegression(max_iter=500), X, y, cv=5).mean() if len(pp) > 20 else np.nan
    # the same contrast with the change times shifted 0.25 s earlier (5 bins): -0.25..0.05 s against -0.55..-0.25 s
    accs = cross_val_score(LogisticRegression(max_iter=500), np.r_[pp[:, 10:16], pp[:, 4:10]], y, cv=5).mean() if len(pp) > 20 else np.nan
    ratio_s = pp[:, 10:16].mean() / pp[:, 4:10].mean()
    clock.append(dict(session=k, ratio_post_pre=post.mean() / pre.mean(), ratio_shifted=ratio_s, decode_true=acc, decode_shifted=accs))
clock = pd.DataFrame(clock)
print(clock.describe().T[["min", "25%", "50%", "max"]])
print("sessions with post/pre ratio <= 1.05:", (clock.ratio_post_pre <= 1.05).sum(), "| sessions where the true-time ratio exceeds the shifted one:", (clock.ratio_post_pre > clock.ratio_shifted).sum(), "of", len(clock))

## Behavioural claims

### Claim blocks-no-reward-in-replay

> Block 0 is the active behavior block and block 5 the passive replay block in all 103 sessions,
> and no lick or reward events occur during passive replay.

In [ ]:
rows = []
for k, (meta, df, arr, ev) in enumerate(OUT):
    pr = meta["presentations"]; b = meta["blocks"]
    b0 = pr[pr.stimulus_block == 0]; b5 = pr[pr.stimulus_block == 5]
    same_seq = len(b0) == len(b5) and np.array_equal(b0.image_name.values, b5.image_name.values)
    in5 = lambda t: int(((t >= ev["p0"]) & (t <= ev["p1"])).sum())
    rows.append(dict(session=k, block0=b[0][0], block5=b[5][0], b0_active=b0.active.mean(), b5_active=b5.active.mean(),
                     trials_in_b0=int(((meta["trials"].start_time >= ev["a0"] - 1) & (meta["trials"].stop_time <= ev["a1"] + 1)).sum()),
                     first_trial_minus_b0=meta["trials"].start_time.min() - ev["a0"], last_trial_minus_b0=meta["trials"].stop_time.max() - ev["a1"],
                     n_trials=len(meta["trials"]), same_image_sequence=same_seq,
                     licks_in_b5=in5(ev["licks"]), rewards_in_b5=in5(ev["rew_all"]),
                     rewards_in_b0=int(((ev["rew_all"] >= ev["a0"]) & (ev["rew_all"] <= ev["a1"] + 1)).sum()), rewards=len(ev["rew_all"]),
                     other_blocks=" ".join(f"{i}:{b[i][0][:12]}" for i in sorted(b) if i not in (0, 5))))
BLK = pd.DataFrame(rows)
print(BLK.block0.str[:40].value_counts().to_dict(), BLK.block5.str[:40].value_counts().to_dict())
print(BLK.other_blocks.value_counts().to_dict())
print("block 0 all active:", (BLK.b0_active == 1).all(), "| block 5 none active:", (BLK.b5_active == 0).all())
print("sessions where every trial lies in block 0 (within 1 s of its bounds):", (BLK.trials_in_b0 == BLK.n_trials).sum(), "of", len(BLK))
print("first trial start minus block 0 start (s): %.2f to %.2f; last trial stop minus block 0 stop: %.2f to %.2f"
      % (BLK.first_trial_minus_b0.min(), BLK.first_trial_minus_b0.max(), BLK.last_trial_minus_b0.min(), BLK.last_trial_minus_b0.max()))
print("sessions with the same image sequence in blocks 0 and 5:", BLK.same_image_sequence.sum())
print("sessions with every reward in block 0:", (BLK.rewards_in_b0 == BLK.rewards).sum())
print("licks in block 5, total:", BLK.licks_in_b5.sum(), "in", (BLK.licks_in_b5 > 0).sum(), "sessions; rewards in block 5:", BLK.rewards_in_b5.sum())
print(BLK[BLK.licks_in_b5 > 0][["session", "licks_in_b5"]].describe().T)

**Result: reproduced.** In all 104 sessions block 0 is the natural-image block with every presentation active, block 5 has none active and the same image sequence, every trial lies in block 0, and no lick or reward time falls in block 5.

### Claim reward-interval-min

> Consecutive rewards are separated by at least 6.96 s, so no peri-reward window holds a neighboring
> reward.

Performance rewards are the reward times not flagged `autorewarded` (M&M 2.1).

In [ ]:
mins = np.array([np.diff(e["earned"]).min() for e in EV])
mins_all = np.array([np.diff(np.sort(e["rew_all"])).min() for e in EV])
mins_win = np.array([np.diff(e["rew"]).min() for e in EV])
print("minimum interval between consecutive performance rewards: %.3f s (per-session minima %.2f to %.2f s, median %.2f)"
      % (mins.min(), mins.min(), mins.max(), np.median(mins)))
print("including automatic rewards: %.3f s; rewards with the whole window in the block: %.3f s" % (mins_all.min(), mins_win.min()))
print("sessions with an interval below 6.96 s:", (mins < 6.96).sum(), "| below 2.5 s (a window overlap):", (mins < 2.5).sum())

**Result: reproduced.** The minimum interval between consecutive performance rewards is 6.972 s (expected 6.96 s), and no session has one below 6.96 s.

### Claim latency-change-to-reward

> Across sessions the median change-to-reward latency is 0.467 s; over 16,330 hit trials the median
> within-session standard deviation is 0.104 s (range 0.046 to 0.174 s), and 3.8% of hits depart
> from their session median by more than 250 ms.

Latency = reward time minus the start time of the trial's change presentation (block 0 stimulus
table). The trial table's `change_time_no_display_delay` is shown as a variant.

In [ ]:
lat = [e["latency"] for e in EV]
lat_nd = [e["hit_reward"] - e["hit_change_nodelay"] for e in EV]
def latsum(L):
    allh = np.concatenate(L); meds = np.array([np.median(x) for x in L]); sds = np.array([np.std(x, ddof=1) for x in L])
    dep = np.concatenate([np.abs(x - np.median(x)) > 0.25 for x in L])
    return dict(hits=len(allh), median_of_session_medians=np.median(meds), pooled_median=np.median(allh),
                median_within_sd=np.median(sds), sd_min=sds.min(), sd_max=sds.max(), pct_over_250ms=100 * dep.mean())
LAT = pd.DataFrame({"change presentation": latsum(lat), "change_time_no_display_delay (variant)": latsum(lat_nd)})
print(LAT.round(5))
print("hit trials in the trial tables:", sum(e["n_hit_trials"] for e in EV), "| earned rewards:", sum(len(e["earned"]) for e in EV))
print("latency range (s): %.3f to %.3f" % (np.concatenate(lat).min(), np.concatenate(lat).max()))
tot = {k: sum(e[k] for e in EV) for k in ("n_hit_all", "n_hit_auto", "n_hit_no_reward", "n_hit_no_change")}
print("hit trials %(n_hit_all)d: auto-rewarded %(n_hit_auto)d, without a reward time %(n_hit_no_reward)d, without a change presentation %(n_hit_no_change)d" % tot)
print("earned rewards whose -0.5 to 2.0 s window lies in the active block: %d of %d" % (sum(len(e["rew"]) for e in EV), sum(len(e["earned"]) for e in EV)))
print("hits per session: min %d, median %.1f, max %d; mean %.1f, so 103 sessions at this rate would hold %.0f" % (
      min(len(x) for x in lat), np.median([len(x) for x in lat]), max(len(x) for x in lat), np.mean([len(x) for x in lat]), 103 * np.mean([len(x) for x in lat])))


**Result: partially reproduced.** With the change time from the trial table's `change_time_no_display_delay`, 15,592 hits (expected 16,330), a median latency of 0.451 s (0.467), and a median within-session SD of 0.111 s (0.104, range 0.046 to 0.177 s) are within tolerance, but 4.20% of hits depart from their session median by more than 250 ms (expected 3.8%, at most 4.18%). With the change presentation's start time the median latency is 0.407 s, outside tolerance. All 15,592 hit trials of the 104 sessions are used (none is auto-rewarded, lacks a reward time or a change presentation, or has its window outside the block); at the mean of 149.9 hits per session, 103 sessions would hold 15,442, so the gap to 16,330 does not come from a filter here.

### Claim lick-reward-coincide

> Water is delivered a median of 9.8 ms after the nearest lick (95th percentile 17.3 ms, all within
> 20 ms), across the 16,330 earned rewards, so licking and reward cannot be separated by alignment.

In [ ]:
pre, absd = [], []
for e in EV:
    L = e["licks"]; r = e["earned"]
    i = np.searchsorted(L, r, side="right") - 1
    p = np.where(i >= 0, r - L[np.clip(i, 0, None)], np.nan)
    j = np.clip(np.searchsorted(L, r), 0, len(L) - 1)
    nxt = L[j] - r
    a = np.where(np.abs(nxt) < np.abs(p), -np.abs(nxt), p)  # signed: reward minus nearest lick
    pre.append(p); absd.append(a)
pre = np.concatenate(pre) * 1e3; absd = np.concatenate(absd) * 1e3
for name, x in (("(a) nearest preceding lick", pre), ("(b) nearest lick in absolute time", absd)):
    x = x[np.isfinite(x)]
    print("%s: n = %d, median %.2f ms, 95th percentile %.2f ms, max %.1f ms, min %.1f ms, within 20 ms %.2f%%, within 22 ms %.2f%%"
          % (name, len(x), np.median(x), np.percentile(x, 95), x.max(), x.min(), 100 * np.mean(np.abs(x) <= 20), 100 * np.mean(np.abs(x) <= 22)))
    print("   rewards with a delay above 20 ms: %d, above 22 ms: %d" % ((np.abs(x) > 20).sum(), (np.abs(x) > 22).sum()))

**Result: partially reproduced.** Water follows the nearest lick by a median of 9.78 ms (expected 9.8) with a 95th percentile of 17.32 ms (17.3), under both readings, but 5 of 15,592 rewards come more than 22 ms after it (maximum 53.5 ms), so not all are within 20 ms.

## The reward-modulated cohort

### Claim fig1e-cohort-size

> Of 54,039 units, 11,032 (20.4%; 32.9% of the 33,502 units testable in both contrasts) form the
> reward-modulated cohort, 7,229 suppressed (65.5%) and 3,803 enhanced (34.5%); 11,271 units were
> significant in both families, of which 239 had discordant signs.

The pipeline of M&M 2.2 to 2.4 (helper `process_session`), with BH at q = 0.05 per area on the
split-half p values pooled across sessions, then per area and family on the Mann-Whitney p values of
split-half positive units responsive in at least one condition. Readings of the open details are
listed in the claim's criterion; the ISI null is run under reading (1a) only (see deviations).

In [ ]:
U["sh_pos"] = vbn.bh_by(U, "p_split", "location", np.isfinite(U.p_split.values))
U["A_sig"], U["A_elig"] = vbn.family_pass(U, "A", U.sh_pos.values)
U["B_sig"], U["B_elig"] = vbn.family_pass(U, "B", U.sh_pos.values)
both = U.A_sig & U.B_sig
conc = both & (np.sign(U.A_si) == np.sign(U.B_si)) & (U.A_si != 0)
U["cohort"] = conc
U["supp"] = conc & (U.A_si < 0)
U["enh"] = conc & (U.A_si > 0)
respA = (U.A_p1 < 0.05) | (U.A_p2 < 0.05); respB = (U.B_p1 < 0.05) | (U.B_p2 < 0.05)
test_a = U.sh_pos & respA & respB
test_b = respA & respB
test_c = U.sh_pos & (U.n_hit > 0) & (U.n_pas > 0) & (U.n_miss > 0)
n = len(U); c = int(conc.sum())
print("units tested: %d | split-half positive: %d (%.1f%%) | r_split undefined: %d" % (n, U.sh_pos.sum(), 100 * U.sh_pos.mean(), U.r_split.isna().sum()))
print("p_split at its minimum 1/(N+1): %d units" % (U.p_split <= 1 / (NPERM + 1) + 1e-12).sum())
for a in A:
    m = U.location == a
    p = np.sort(U.p_split[m & U.sh_pos])
    print("  %s: split-half positive %d of %d, largest passing p %.4f" % (a, (m & U.sh_pos).sum(), m.sum(), p.max() if len(p) else np.nan))
print("Family A significant: %d (eligible %d) | Family B significant: %d (eligible %d)" % (U.A_sig.sum(), U.A_elig.sum(), U.B_sig.sum(), U.B_elig.sum()))
print("significant in both families: %d | discordant signs: %d | cohort: %d (%.1f%% of tested)" % (both.sum(), (both & ~conc).sum(), c, 100 * c / n))
print("suppressed: %d (%.1f%%) | enhanced: %d (%.1f%%)" % (U.supp.sum(), 100 * U.supp.sum() / c, U.enh.sum(), 100 * U.enh.sum() / c))
for nm, t in (("(a) split-half positive, responsive in both families", test_a), ("(b) responsive in both families", test_b),
              ("(c) split-half positive with trials in all three conditions", test_c)):
    print("testable %s: %d; cohort is %.1f%% of it" % (nm, t.sum(), 100 * c / t.sum()))
print("by session: cohort units per session min %d, median %d, max %d" % tuple(U.groupby("session").cohort.sum().agg(["min", "median", "max"])))# reading (1b) of the ISI null: ISIs shuffled over the whole recording (verifier round 1). For all 104 sessions it
# takes over an hour (a first run of this notebook stopped at the 60-minute cell limit), so it is run on 12 sessions
# spread over the list, and both readings are compared there, with BH per area over those sessions' units.
KREC = list(range(0, len(SEL), 9))
t = time.time()
PREC = vbn.run_rec_null(SEL.asset_id.tolist(), KREC, NPERM, seed("fig1e-cohort-size"), workers=6)
U["p_split_rec"] = np.nan
for k, p in PREC.items():
    U.loc[U.session == k, "p_split_rec"] = p
sub = U.session.isin(KREC).values
print("reading (1b) on sessions %s: %d units, %.0f s" % (KREC, sub.sum(), time.time() - t))
def cohort_of(p_col):
    shpos = vbn.bh_by(U, p_col, "location", sub & np.isfinite(U[p_col].values))
    a_sig, _ = vbn.family_pass(U, "A", shpos); b_sig, _ = vbn.family_pass(U, "B", shpos)
    bth = a_sig & b_sig; cc = bth & (np.sign(U.A_si) == np.sign(U.B_si)).values & (U.A_si != 0).values
    ta = shpos & respA.values & respB.values
    return dict(units=int(sub.sum()), split_half_positive=int(shpos.sum()), A=int(a_sig.sum()), B=int(b_sig.sum()), both=int(bth.sum()),
                discordant=int((bth & ~cc).sum()), cohort=int(cc.sum()), pct_of_units=round(100 * cc.sum() / sub.sum(), 1),
                pct_suppressed=round(100 * (cc & (U.A_si < 0).values).sum() / cc.sum(), 1), testable_a=int(ta.sum())), shpos, cc
r1a, s1a, c1a = cohort_of("p_split"); r1b, s1b, c1b = cohort_of("p_split_rec")
print(pd.DataFrame({"null (1a), active block": r1a, "null (1b), whole recording": r1b}).T.to_string())
print("on these sessions: split-half positive under both %d, only (1a) %d, only (1b) %d; cohort under both %d, only (1a) %d, only (1b) %d"
      % ((s1a & s1b).sum(), (s1a & ~s1b).sum(), (~s1a & s1b).sum(), (c1a & c1b).sum(), (c1a & ~c1b).sum(), (~c1a & c1b).sum()))
print("Spearman of the two p values: %.3f" % stats.spearmanr(U.p_split[sub], U.p_split_rec[sub], nan_policy="omit")[0])


Sensitivity of the split-half gate to the number of permutations, on 100 units spread over each of three sessions (the first, the 52nd and the last): 1,000
permutations (the paper's number) against the 100 used here, under the same claim seed plus 500. Running 1,000 for all 54,045 units would take about nine CPU hours.

In [ ]:
rng = np.random.default_rng(seed("fig1e-cohort-size") + 500)
t = time.time(); parts = []
for k0 in (0, 51, 103):
    meta0, df0, arr0, ev0 = OUT[k0]
    f0 = vbn.open_nwb(SEL.asset_id[k0]); st_all = f0["units/spike_times"]
    evs = ev0["rew"] + vbn.WIN[0]; nb = 250; odd = (np.arange(len(ev0["rew"])) % 2).astype(np.int64); lut = vbn.make_lut(evs, nb)
    df0 = df0.iloc[np.linspace(0, len(df0) - 1, 100).astype(int)]  # 100 units spread over the session's probes
    p1000 = []
    for r in df0.itertuples():
        s = st_all[int(r.st_start):int(r.st_stop)]; s = s[(s >= ev0["a0"]) & (s <= ev0["a1"])]
        nl = vbn.isi_null(s, evs, nb, odd, 1000, rng, lut=lut)
        p1000.append((1 + np.sum(nl >= r.r_split)) / 1001 if np.isfinite(r.r_split) else np.nan)
    f0.close()
    parts.append(df0.assign(p1000=p1000))
    d_ = parts[-1]
    print("session %d: 100 units; p <= 0.05 with 100 permutations %d, with 1,000 %d, both %d" % (k0, (d_.p_split <= 0.05).sum(), (d_.p1000 <= 0.05).sum(), ((d_.p_split <= 0.05) & (d_.p1000 <= 0.05)).sum()))
df0 = pd.concat(parts)
print("three sessions: %d units, %.0f s" % (len(df0), time.time() - t))
print("p <= 0.05 with 100 permutations: %d; with 1,000: %d; both: %d" % ((df0.p_split <= 0.05).sum(), (df0.p1000 <= 0.05).sum(), ((df0.p_split <= 0.05) & (df0.p1000 <= 0.05)).sum()))
print("units at the minimum p: 100 perms %d (p = %.4f); 1,000 perms %d (p = %.4f)" % ((df0.p_split <= 1 / 101 + 1e-12).sum(), 1 / 101, (df0.p1000 <= 1 / 1001 + 1e-12).sum(), 1 / 1001))
print("Spearman of the two p values: %.3f" % stats.spearmanr(df0.p_split, df0.p1000, nan_policy="omit")[0])

**Result: partially reproduced, resting on an assumption.** The cohort holds 10,822 units (expected 11,032), 20.0% of the 54,045 tested, 7,378 suppressed (68.2%) and 3,444 enhanced; 11,108 units are significant in both families and 34,959 are testable under reading (a), all within tolerance. The 286 units with discordant signs are outside the tolerance of 239 (215 to 263). The split-half null used 100 permutations: on 300 units of three sessions, 225 units have p <= 0.05 with 100 permutations and 229 with 1,000 (224 under both). Reading (1b) of the null, over the whole recording, was run on 12 of the 104 sessions (6,553 units): the cohort there is 1,268 units against 1,272 under reading (1a), 1,266 in both, with 19 discordant under each; 8 units are in only one of the two cohorts (6 only under 1a, 2 only under 1b), and 74 and 63 units are split-half positive under only one reading, so the reading changes the counts a little and the outcome not at all.

### Claim fig1d-split-half-above-null

> The cohort's split-half reliabilities lie well above the ISI-shuffle null in all six areas.

In [ ]:
NUL = np.concatenate([o[2]["nulls"] for o in OUT])
assert len(NUL) == len(U)
rows = []
for a in A:
    m = (U.location == a).values & U.cohort.values
    obs = U.r_split.values[m]; nl = NUL[m].ravel(); nl = nl[np.isfinite(nl)]
    own_med = np.nanmedian(NUL[m], axis=1)
    rows.append(dict(area=a, cohort=m.sum(), median_r=np.median(obs), min_r=obs.min(), null_median=np.median(nl), null_p95=np.percentile(nl, 95),
                     null_max=nl.max(), reading_a=np.median(obs) > np.percentile(nl, 95), pct_above_own_null_median=100 * np.mean(obs > own_med),
                     reading_b=bool(np.all(obs > own_med))))
F1D = pd.DataFrame(rows); print(F1D.round(3).to_string())

**Result: reproduced, resting on an assumption.** In every area the cohort's median r_split (0.853 to 0.902) exceeds the 95th percentile of the pooled null (0.380 to 0.409), and every cohort unit's r_split exceeds the median of its own null; both readings hold. Reading (b) cannot fail here: every cohort unit passed the split-half gate at BH-corrected p_split <= 0.0396, so at most 3 of its 100 null values reach its r_split, and its r_split must exceed its null's median. Reading (a) carries the claim.

### Claim fig1c-example-cells

> The cohort contains reward-enhanced units in VISam and reward-suppressed units in VISrl with
> near-ceiling split-half reproducibility (r_split 1.00 and 0.99 in the two examples).

In [ ]:
e_am = U[U.enh & (U.location == "VISam")].r_split; s_rl = U[U.supp & (U.location == "VISrl")].r_split
print("enhanced VISam cohort units: %d, max r_split %.4f, with r_split >= 0.99: %d" % (len(e_am), e_am.max(), (e_am >= 0.99).sum()))
print("suppressed VISrl cohort units: %d, max r_split %.4f, with r_split >= 0.98: %d" % (len(s_rl), s_rl.max(), (s_rl >= 0.98).sum()))

**Result: reproduced.** The cohort holds 8 enhanced VISam units with r_split >= 0.99 (maximum 0.9963) and 32 suppressed VISrl units with r_split >= 0.98 (maximum 0.9960).

### Claim fig1e-suppressed-fraction-by-area

> The per-unit suppressed fraction declines along the functional hierarchy, 86% in VISp (n = 2,374),
> 68% in VISl (1,782), 60% in VISrl (1,894), 61% in VISal (1,511), 59% in VISpm (1,765) and 51% in
> VISam (1,706), so the enhanced fraction rises from 14% in VISp to 49% in VISam.

In [ ]:
paper = {"VISp": (86, 2374), "VISl": (68, 1782), "VISrl": (60, 1894), "VISal": (61, 1511), "VISpm": (59, 1765), "VISam": (51, 1706)}
rows = []
for a in A:
    m = U.cohort & (U.location == a)
    rows.append(dict(area=a, cohort=m.sum(), paper_n=paper[a][1], suppressed=(m & U.supp).sum(), supp_pct=100 * (m & U.supp).sum() / m.sum(), paper_pct=paper[a][0],
                     within_tol=abs(100 * (m & U.supp).sum() / m.sum() - paper[a][0]) <= paper[a][0] * 0.1))
F1E = pd.DataFrame(rows); print(F1E.round(1).to_string())
print("highest:", F1E.loc[F1E.supp_pct.idxmax(), "area"], "| lowest:", F1E.loc[F1E.supp_pct.idxmin(), "area"])
for k in sorted(set(U.subject)):
    pass
bys = U[U.cohort].groupby(["subject", "location"]).supp.mean().unstack()[A]
print("per mouse (suppressed fraction of its cohort units; first 10 mice):"); print(bys.head(10).round(2))
print("mice with VISp above VISam (both present):", int((bys.VISp > bys.VISam).sum()), "of", int(bys[["VISp", "VISam"]].notna().all(1).sum()))

**Result: reproduced.** The suppressed fractions are 87.1% in VISp, 70.3% in VISl, 62.6% in VISrl, 63.0% in VISal, 63.3% in VISpm and 53.6% in VISam, all within tolerance, with VISp highest and VISam lowest. VISp has 2,421 cohort units (2,108 suppressed) and VISam 1,637, within tolerance of 2,374, 2,048 and 1,706. VISp exceeds VISam in 45 of 54 mice.

### Claim fig1l-session-gradient

> With the session as the unit of analysis, the median suppressed fraction falls from 0.92 in VISp to
> 0.52 in VISam and the areas differ (Kruskal-Wallis, p < 0.001).

In [ ]:
g = U[U.cohort].groupby(["session", "location"]).supp.agg(["mean", "size"]).reset_index()
g5 = g[g["size"] >= 5]
per = {a: g5[g5.location == a]["mean"].values for a in A}
for a in A:
    print("%s: sessions with >= 5 cohort units %d, median suppressed fraction %.3f" % (a, len(per[a]), np.median(per[a])))
kw = stats.kruskal(*[per[a] for a in A])
print("Kruskal-Wallis H = %.1f, p = %.3g" % kw)

**Result: partially reproduced.** With sessions holding at least five cohort units, the median suppressed fraction is 0.911 in VISp (expected 0.92) and 0.575 in VISam (expected 0.52, tolerance up to 0.572), and the areas differ (Kruskal-Wallis p = 3.8e-17).

### Claim latency-jitter-control

> Drawing pseudo-reward latencies at random from each session's hit latencies, twenty times, leaves
> 99.2% of the cohort passing Family A with unchanged SI sign (range 99.1 to 99.3%), and the index
> correlates with its original value at rho = 0.996.

Each realization re-runs Family A (responsiveness, Mann-Whitney, BH per area over split-half
positive units) with the jittered latencies (helper, `J0` to `J19`; generator seeded with the
claim's seed plus the session's index).

In [ ]:
coh = U.cohort.values
rows = []
for j in range(20):
    sig, _ = vbn.family_pass(U, f"J{j}", U.sh_pos.values)
    same = sig & (np.sign(U[f"J{j}_si"].values) == np.sign(U.A_si.values))
    rho = stats.spearmanr(U[f"J{j}_si"].values[coh], U.A_si.values[coh], nan_policy="omit")[0]
    r = stats.pearsonr(U[f"J{j}_si"].values[coh], U.A_si.values[coh])[0]
    rows.append(dict(realization=j, pct_pass_same_sign=100 * same[coh].mean(), spearman=rho, pearson=r))
JIT = pd.DataFrame(rows)
print(JIT.describe().T[["mean", "min", "max"]].round(4))

**Result: reproduced.** Over twenty jittered realizations, 99.14% of the cohort (range 98.97 to 99.23%) still passes Family A with the same SI sign (expected 99.2%), and the jittered SI correlates with the original at a mean Spearman rho of 0.9959 (0.996); Pearson r is 0.9995.

## Waveform classes

### Claim waveform-ambiguous-band

> The two-component Gaussian mixture on trough-to-peak duration labels 849 of 53,806 units (1.6%)
> ambiguous, 227 of them in the cohort (2.1%); widening the band to 0.3 to 0.7 removes 1,280 units
> (2.4%) and leaves the suppressed fractions at 0.790 in narrow and 0.617 in broad units, against
> 0.790 and 0.619 with the 0.4 to 0.6 band.

Duration readings: (a) from the peak-channel mean waveform (`waveform_mean`, 30 kHz samples); (b)
the units table's `waveform_duration`. Mixture: two components, full covariance, eight
initialisations (M&M 2.5), seeded with the claim's seed.

In [ ]:
from sklearn.mixture import GaussianMixture
def classify(x, sd):
    ok = np.isfinite(x) & (x > 0)
    gm = GaussianMixture(2, covariance_type="full", n_init=8, random_state=sd).fit(x[ok, None])
    nar = int(np.argmin(gm.means_.ravel()))
    post = np.full(len(x), np.nan); post[ok] = gm.predict_proba(x[ok, None])[:, nar]
    return ok, post, gm
WF = {}
for nm, col in (("a", "wf_dur"), ("a_up", "wf_dur_up"), ("b", "waveform_duration")):
    rows = []
    for s_off in range(5):
        ok, post, gm = classify(U[col].values, seed("waveform-ambiguous-band") + 1000 * s_off)
        amb4 = ok & (post >= 0.4) & (post <= 0.6); amb3 = ok & (post >= 0.3) & (post <= 0.7)
        rows.append(dict(seed_offset=1000 * s_off, fitted=ok.sum(), log_lik=round(float(gm.score(U[col].values[ok, None]) * ok.sum()), 1), ambiguous_04_06=amb4.sum(), amb_cohort=(amb4 & coh).sum(), ambiguous_03_07=amb3.sum(),
                         means_ms=np.round(np.sort(gm.means_.ravel()), 3).tolist(),
                         boundary_ms=float(np.linspace(0.05, 2.5, 4901)[np.argmin(np.abs(gm.predict_proba(np.linspace(0.05, 2.5, 4901)[:, None])[:, int(np.argmin(gm.means_.ravel()))] - 0.5))])))
        if s_off == 0:
            WF[nm] = post
            U[f"post_{nm}"] = post
    print("reading (%s), duration from %s:" % (nm, col)); print(pd.DataFrame(rows).to_string())
    ok = np.isfinite(WF[nm]); amb4 = ok & (WF[nm] >= 0.4) & (WF[nm] <= 0.6); amb3 = ok & (WF[nm] >= 0.3) & (WF[nm] <= 0.7)
    print("  %.2f%% of fitted ambiguous (0.4-0.6), %.2f%% of the cohort; %.2f%% ambiguous with 0.3-0.7" % (100 * amb4.sum() / ok.sum(), 100 * (amb4 & coh).sum() / coh.sum(), 100 * amb3.sum() / ok.sum()))
    for band, (lo, hi) in (("0.4-0.6", (0.4, 0.6)), ("0.3-0.7", (0.3, 0.7))):
        nar = coh & ok & (WF[nm] > hi); bro = coh & ok & (WF[nm] < lo)
        print("  band %s: cohort narrow %d, broad %d; suppressed fraction narrow %.3f, broad %.3f" % (band, nar.sum(), bro.sum(), U.supp[nar].mean(), U.supp[bro].mean()))
for col in ("wf_dur", "wf_dur_up", "waveform_duration"):
    print("%s quantiles (ms) 1, 10, 25, 50, 75, 90, 99%%:" % col, np.round(np.nanpercentile(U[col], [1, 10, 25, 50, 75, 90, 99]), 3).tolist(),
          "| distinct values:", U[col].nunique())
print("duration from the mean waveform against the stored one: Pearson r = %.3f, median difference %.3f ms"
      % (np.corrcoef(U.wf_dur.fillna(0), U.waveform_duration.fillna(0))[0, 1], np.nanmedian(U.wf_dur - U.waveform_duration)))
hist, edges = np.histogram(U.wf_dur.dropna(), bins=np.arange(0, 2.1, 0.1))
print("histogram of wf_dur (0.1 ms bins from 0): %s; above 2.0 ms %d" % (hist.tolist(), int((U.wf_dur >= 2.0).sum())))
hist, edges = np.histogram(U.waveform_duration.dropna(), bins=np.arange(0, 2.1, 0.1))
print("histogram of waveform_duration (0.1 ms bins from 0): %s; above 2.0 ms %d" % (hist.tolist(), int((U.waveform_duration >= 2.0).sum())))
for col in ("wf_dur", "wf_dur_up"):
    dd = (U[col] - U.waveform_duration).abs()
    print("%s within 0.02 ms of the stored duration: %.1f%% of units; more than 0.1 ms away: %d units (their stored median %.3f, %s median %.3f ms)"
          % (col, 100 * (dd <= 0.02).mean(), (dd > 0.1).sum(), U.waveform_duration[dd > 0.1].median(), col, U[col][dd > 0.1].median()))


In [ ]:
# layout of waveform_mean (verifier round 1): rows per unit, and whether the row with the largest peak-to-trough
# amplitude is the row of the unit's peak channel (probe_channel_number), on 100 units of each of three sessions
print("waveform_mean rows per unit, all analysed units:", (U.wf_stop - U.wf_start).value_counts().to_dict())
rows = []; nsamp = set()
for k0 in (0, 51, 103):
    f0 = vbn.open_nwb(SEL.asset_id[k0]); W0 = f0["units/waveform_mean"]; nsamp.add(W0.shape[1])
    d_ = U[U.session == k0]; d_ = d_.iloc[np.linspace(0, len(d_) - 1, 100).astype(int)]
    for r in d_.itertuples():
        w = W0[int(r.wf_start):int(r.wf_stop)]; amp = w.max(1) - w.min(1)
        rows.append(dict(session=k0, peak=int(r.probe_channel_number), argmax=int(np.argmax(amp)), ratio=amp[int(r.probe_channel_number)] / amp.max()))
    f0.close()
L_ = pd.DataFrame(rows)
print("samples per waveform_mean row in these three sessions:", sorted(nsamp))
print("units checked %d: largest-amplitude row equals probe_channel_number in %d, within 4 channels in %d; amplitude on the peak-channel row / largest: median %.3f, 5th percentile %.3f"
      % (len(L_), (L_.peak == L_.argmax).sum(), ((L_.peak - L_.argmax).abs() <= 4).sum(), L_.ratio.median(), L_.ratio.quantile(0.05)))


**Result: not reproduced.** With the stored `waveform_duration` (reading b), the mixture's components have means 0.264 and 0.621 ms and the band 0.4 to 0.6 labels 405 of 54,045 units ambiguous (expected 849), 110 in the cohort (227); the band 0.3 to 0.7 labels 1,205 (1,280), within tolerance. With durations from the 30 kHz mean waveform (reading a), which take 60 distinct values, the mixture does not separate narrow from broad units: under the claim's seed and the seed plus 1000 its components have means 0.534 and 0.616 ms (log-likelihood 24,671 and 24,662), under the other three seeds 0.57 and 1.425 ms (19,781), so the claim's seed gives the better fit, and it labels 1,292 units ambiguous. The waveform rows are laid out as assumed: every unit has 384 rows, and in 230 of 300 units checked the largest-amplitude row is the peak channel's (299 within 4 channels). Upsampling the waveform to 200 points (a variant) brings 97.5% of durations within 0.02 ms of the stored ones, but the mixture again puts one component on the long tail (means 0.571 and 1.491 ms; 34 ambiguous units). The tail is real: 1,263 units have an upsampled duration more than 0.1 ms from the stored one (median 1.421 against 0.371 ms), so for these units the stored duration is not the trough to the following maximum of the peak-channel mean waveform, and reading (a) as the Methods define it does not give two classes.

In [ ]:
# class labels used by the claims below: reading (b), the stored duration, unless said; band 0.4 to 0.6
def classes(nm="b", lo=0.4, hi=0.6):
    p = U[f"post_{nm}"].values
    return np.isfinite(p) & (p > hi), np.isfinite(p) & (p < lo)
NAR, BRO = classes("b")

### Claim fig1jk-narrow-fraction-by-area

> The narrow-spiking fraction of the cohort is highest in VISp and VISrl (about 26%) and lower in the
> other four areas (14 to 19%).

In [ ]:
rows = []
for nm in ("a", "b"):
    nar, bro = classes(nm)
    for a in A:
        m = coh & (U.location == a).values
        r = dict(duration=nm, area=a, all_cohort=100 * nar[m].sum() / (nar[m].sum() + bro[m].sum()))
        for lab, sub in (("suppressed", U.supp.values), ("enhanced", U.enh.values)):
            mm = m & sub
            r[lab] = 100 * nar[mm].sum() / (nar[mm].sum() + bro[mm].sum())
        rows.append(r)
F1JK = pd.DataFrame(rows); print(F1JK.round(1).to_string())

**Result: reproduced.** With the stored duration (reading b) and the cohort's narrow and broad units, the narrow fraction is 25.0% in VISp and 27.1% in VISrl (expected about 26%), and 15.2 to 20.1% in the other four areas, inside 12.6 to 20.9%. Durations from the mean waveform (reading a) give 28.5 to 44.2%.

### Claim fig1h-median-isi-narrow-broad

> Median ISI is lower in narrow-spiking cohort units (40.9 ms, n = 2,308) than in broad-spiking units
> (85.0 ms, n = 8,716).

In [ ]:
rows = []
for nm in ("a", "b"):
    p = U[f"post_{nm}"].values
    for cls_nm, (nar, bro) in (("ambiguous excluded", classes(nm)), ("larger posterior", (np.isfinite(p) & (p >= 0.5), np.isfinite(p) & (p < 0.5)))):
        for span in ("median_isi_all", "median_isi_active"):
            x = U[span].values * 1e3
            rows.append(dict(duration=nm, classes=cls_nm, span=span, n_narrow=(coh & nar).sum(), n_broad=(coh & bro).sum(),
                             isi_narrow_ms=np.nanmedian(x[coh & nar]), isi_broad_ms=np.nanmedian(x[coh & bro])))
F1H = pd.DataFrame(rows); print(F1H.round(1).to_string())

**Result: reproduced.** With the stored duration, ambiguous units excluded and ISIs over the whole recording, the median ISI is 42.2 ms in 2,245 narrow cohort units and 88.0 ms in 8,467 broad ones (expected 40.9 ms, 2,308; 85.0 ms, 8,716).

### Claim fig1gi-waveform-separation

> The narrow and broad populations of the cohort separate also in waveform half-width and in spiking
> statistics (median ISI, autocorrelogram rise time).

The autocorrelogram rise time is not defined in the paper and is not computed here.

In [ ]:
for nm in ("a", "b"):
    nar, bro = classes(nm)
    hw_n, hw_b = U.wf_hw.values[coh & nar], U.wf_hw.values[coh & bro]
    is_n, is_b = U.median_isi_all.values[coh & nar] * 1e3, U.median_isi_all.values[coh & bro] * 1e3
    print("duration (%s): half-width median narrow %.3f ms, broad %.3f ms (Mann-Whitney p = %.2g); median ISI narrow %.1f ms, broad %.1f ms (p = %.2g)"
          % (nm, np.nanmedian(hw_n), np.nanmedian(hw_b), stats.mannwhitneyu(hw_n[np.isfinite(hw_n)], hw_b[np.isfinite(hw_b)]).pvalue,
             np.nanmedian(is_n), np.nanmedian(is_b), stats.mannwhitneyu(is_n[np.isfinite(is_n)], is_b[np.isfinite(is_b)]).pvalue))
print("half-width range (ms): %.3f to %.3f; undefined for %d cohort units" % (np.nanmin(U.wf_hw[coh]), np.nanmax(U.wf_hw[coh]), np.isnan(U.wf_hw[coh]).sum()))

**Result: reproduced.** Narrow cohort units have a lower median half-width (0.149 against 0.204 ms with the stored duration; 0.170 against 0.203 ms with reading a) and a lower median ISI (42.2 against 88.0 ms; 54.7 against 89.5 ms); Mann-Whitney p < 1e-190 for each, as a variant. The autocorrelogram rise time was not computed.

### Claim narrow-more-suppressed

> Narrow-spiking cohort units are suppressed more often than broad-spiking units (suppressed fraction
> 0.790 against 0.619).

In [ ]:
for nm in ("a", "b"):
    for lo, hi in ((0.4, 0.6), (0.3, 0.7)):
        nar, bro = classes(nm, lo, hi)
        print("duration (%s), band %.1f-%.1f: suppressed fraction narrow %.3f (n = %d), broad %.3f (n = %d)"
              % (nm, lo, hi, U.supp.values[coh & nar].mean(), (coh & nar).sum(), U.supp.values[coh & bro].mean(), (coh & bro).sum()))

**Result: reproduced.** With the stored duration the suppressed fraction is 0.815 in narrow and 0.647 in broad cohort units (band 0.4 to 0.6), and 0.647 in broad units with the band 0.3 to 0.7, all within tolerance (expected 0.790, 0.619 and 0.617). Reading (a) gives 0.748, 0.650 and 0.647, also within tolerance.

## Reward modulation index

### Claim fig1mn-modulation-index-negative

> The reward modulation index M has a negative median in all six areas; over the cohort its median is
> -0.1323 and 71.6% of units have M < 0.

M = (r_post - r_pre) / (r_post + r_pre) on the active-block reward-aligned histogram (M&M 2.5),
r_post over 0 to +2.0 s, baselines as listed. The stored histogram spans -2.1 to +2.1 s in 10 ms
bins.

In [ ]:
PS = np.concatenate([o[2]["psth"] for o in OUT]).astype(float)
nrew = np.array([OUT[k][2]["n_rew"] for k in U.session.values])
RATE = PS / (nrew[:, None] * vbn.BIN)
RATE_SM = vbn.smooth(RATE)
tb = vbn.WIDE[0] + vbn.BIN * np.arange(PS.shape[1])  # bin starts
def wmean(R, lo, hi):
    m = (tb >= lo - 1e-9) & (tb < hi - 1e-9)
    return R[:, m].mean(1)
def mod_index(R, base):
    post = wmean(R, 0.0, 2.0); pre = wmean(R, *base)
    with np.errstate(all="ignore"):
        return np.where(post + pre > 0, (post - pre) / (post + pre), np.nan)
BASES = [(-0.5, 0.0), (-1.0, -0.5), (-1.5, -1.0), (-2.0, -1.5)]
MI = {(sm, b): mod_index(R, b) for sm, R in (("smoothed", RATE_SM), ("unsmoothed", RATE)) for b in BASES}
for sm in ("smoothed", "unsmoothed"):
    m = MI[(sm, BASES[0])][coh]
    print("%s, baseline -0.5 to 0: median M %.4f, %% M < 0 = %.1f, undefined %d; per area medians: %s"
          % (sm, np.nanmedian(m), 100 * np.nanmean(m < 0), np.isnan(m).sum(),
             {a: round(float(np.nanmedian(MI[(sm, BASES[0])][coh & (U.location == a).values])), 4) for a in A}))
print("M range over the cohort: %.3f to %.3f" % (np.nanmin(MI[("smoothed", BASES[0])][coh]), np.nanmax(MI[("smoothed", BASES[0])][coh])))

**Result: partially reproduced.** The median M is negative in all six areas (-0.278 in VISp to -0.111 in VISrl) and 75.3% of cohort units have M < 0 (expected 71.6%), but the cohort median M is -0.1743, outside the tolerance of -0.1323 (-0.1455 to -0.1191). The unsmoothed histogram gives -0.1757 and 75.2%.

### Claim fig1m-narrow-stronger-m

> Narrow-waveform units are more strongly suppressed than broad-waveform units, with a more negative
> reward modulation index.

In [ ]:
for sm in ("smoothed", "unsmoothed"):
    M0 = MI[(sm, BASES[0])]
    for nm in ("a", "b"):
        nar, bro = classes(nm)
        x, y = M0[coh & nar], M0[coh & bro]
        print("%s, duration (%s): median M narrow %.4f (n = %d), broad %.4f (n = %d); Mann-Whitney p = %.2g"
              % (sm, nm, np.nanmedian(x), np.isfinite(x).sum(), np.nanmedian(y), np.isfinite(y).sum(), stats.mannwhitneyu(x[np.isfinite(x)], y[np.isfinite(y)]).pvalue))

**Result: reproduced.** Narrow cohort units have a more negative median M than broad ones: -0.2276 against -0.1570 with the stored duration and -0.1982 against -0.1616 with reading (a); Mann-Whitney p = 1.6e-30 and 4.3e-10, as variants.

### Claim m-baseline-sensitivity

> Moving the M baseline to -1.5 to -1.0 s raises the median M from -0.1323 to -0.0351 and lowers the
> fraction with M < 0 from 71.6% to 54.5%; the median changes sign in VISal, VISpm and VISam, VISp
> stays the most negative under every baseline, and 44.6% of reward-enhanced units have negative M
> under the adjacent baseline against 1.9% under the retreated one.

In [ ]:
enh = U.enh.values
for sm in ("smoothed", "unsmoothed"):
    rows = []
    for b in BASES:
        M0 = MI[(sm, b)]
        r = dict(baseline=f"{b[0]} to {b[1]}", median_M=np.nanmedian(M0[coh]), pct_neg=100 * np.nanmean(M0[coh] < 0), pct_enh_neg=100 * np.nanmean(M0[enh] < 0))
        for a in A:
            r[a] = np.nanmedian(M0[coh & (U.location == a).values])
        r["most_negative"] = min(A, key=lambda a: r[a])
        rows.append(r)
    print(sm); print(pd.DataFrame(rows).round(4).to_string())
print("rewards with the -2.0 s baseline before the block start:", sum(int(((e["rew"] - 2.0) < e["a0"]).sum()) for e in EV))

**Result: partially reproduced.** With the baseline at -1.5 to -1.0 s, 55.9% of cohort units have M < 0 (expected 54.5%), and 42.7% of enhanced units have M < 0 with the adjacent baseline (44.6%), within tolerance; VISp is the most negative area under all four baselines and stays negative (-0.083). The median M is -0.0413 (expected -0.0351, tolerance to -0.0386), 2.41% of enhanced units have M < 0 under the retreated baseline (expected 1.9%), and the median changes sign in VISal (+0.002) and VISam (+0.038) but not VISpm (-0.002).

## Cortical layers

Layers from the Allen CCFv3 25 um annotation volume at each unit's CCF coordinates (the electrodes
table's x, y, z of the unit's peak channel), the structure mapped to its laminar acronym (M&M 2.5).
The volume is an outside input: BrainGlobe's packaging of the CCFv3 annotation (`allen_mouse_25um`
v1.2, GIN repository brainglobe/atlases at commit 78272d68, file checked by its SHA-256), or, when GIN
refuses the request (it answered 403 to this runner on 2026-11-21), the same CCFv3 2017 annotation from
the Allen Institute's download with the Allen structure ontology, each checked by its SHA-256. The
cell prints which source this run used and, when GIN answers, loads the Allen files as well and
compares the two volumes, their structure lists and the layers they give each unit. A structure carries a
layer only when its name names one, so that hippocampal fields such as CA1 never read as layer 1. These
claims are `data: with_inputs`.

In [ ]:
t = time.time()
ANN, ACR, ATLAS_META, ATLAS_SHA = vbn.load_atlas()
print("atlas", ATLAS_META["name"], ATLAS_META["version"], ATLAS_META["orientation"], ATLAS_META["shape"], ATLAS_META["resolution"], "sha256", ATLAS_SHA, "| %.0f s" % (time.time() - t))
U["ccf_area"], U["layer"], U["bdist"] = vbn.unit_layers(ANN, ACR, U.x.values, U.y.values, U.z.values)
# the same annotation from the other source (verifier round 2): whether the two give the same layers in this run
if ATLAS_META["name"].startswith("allen_mouse_25um (Allen"):
    print("GIN refused the request, so BrainGlobe's file could not be compared with the Allen files in this run")
else:
    t = time.time()
    ANN2, ACR2, META2, SHA2 = vbn._load_atlas_allen()
    print("second source:", META2["name"], META2["shape"], "sha256", SHA2, "| %.0f s" % (time.time() - t))
    print("voxels with the same structure id in both: %d of %d" % ((ANN == ANN2).sum(), ANN.size))
    ids_ = np.union1d(np.unique(ANN), np.unique(ANN2))
    print("structure ids in the volumes: %d; with a different acronym or laminar flag in the two lists: %d"
          % (len(ids_), sum(ACR.get(int(i)) != ACR2.get(int(i)) for i in ids_)))
    a2, l2, d2 = vbn.unit_layers(ANN2, ACR2, U.x.values, U.y.values, U.z.values)
    same = ((pd.Series(a2).fillna("-").values == U.ccf_area.fillna("-").values) & (pd.Series(l2).fillna("-").values == U.layer.fillna("-").values)
            & np.isclose(d2, U.bdist.values, equal_nan=True))
    print("units with the same CCF area, layer and boundary distance under both sources: %d of %d" % (same.sum(), len(U)))
    del ANN2, a2, l2, d2
del ANN
U["lgroup"] = U.layer.map(vbn.LAYER_GROUP)
U["rate_r1"] = U.n_spikes / (U.t_last - U.t_first)  # whole-recording rate, reading (r1) of claim fig2c-cv-lower
haslayer = U.layer.notna().values
# stricter variant (verifier round 1): a layer only where the CCF structure is a visual area
haslayer_vis = haslayer & U.ccf_area.fillna("").str.startswith("VIS").values
print("units in structures whose acronym ends in a layer suffix but whose name names no layer (no layer given):",
      U[~haslayer & U.ccf_area.fillna("").str.match(r".*(1|2/3|4|5|6a|6b)$").values].ccf_area.value_counts().to_dict())
print("CCF structures carrying a layer:", U[haslayer].ccf_area.value_counts().head(12).to_dict())
print("units with a layer:", haslayer.sum(), "| by layer:", U.layer.value_counts().to_dict())
nonpos = ((U.x <= 0) | (U.y <= 0) | (U.z <= 0)).values
print("units with a non-positive CCF coordinate:", int(nonpos.sum()), "| of them without a layer:", int((nonpos & ~haslayer).sum()))

### Claim layer-assignment-coverage

> A layer is assigned to 87.5% of the 54,039 units and 84.9% of the cohort, from 96.6% in VISam to
> 48.0% in VISrl, and the CCF-derived area agrees with the unit-table label for 63.5% of tested
> units.

In [ ]:
print("layer assigned: %.1f%% of tested units, %.1f%% of the cohort" % (100 * haslayer.mean(), 100 * haslayer[coh].mean()))
rows = [dict(area=a, all_units=100 * haslayer[(U.location == a).values].mean(), cohort=100 * haslayer[coh & (U.location == a).values].mean()) for a in A]
print(pd.DataFrame(rows).round(1).to_string())
print("CCF-derived area equal to the unit-table label: %.1f%% of tested units (%.1f%% of those with a layer)"
      % (100 * (U.ccf_area == U.location).mean(), 100 * (U.ccf_area == U.location)[haslayer].mean()))
print("stricter variant, layer only in a CCF visual area: %.1f%% of tested units, %.1f%% of the cohort; by area (all units) %s"
      % (100 * haslayer_vis.mean(), 100 * haslayer_vis[coh].mean(), {a: round(100 * haslayer_vis[(U.location == a).values].mean(), 1) for a in A}))
same = (U.ccf_area == U.location).values
print("stricter variant, layer only where the CCF area equals the unit-table area: %.1f%% of tested units, %.1f%% of the cohort; by area (all units) %s"
      % (100 * (haslayer & same).mean(), 100 * (haslayer & same)[coh].mean(), {a: round(100 * (haslayer & same)[(U.location == a).values].mean(), 1) for a in A}))
print("most common CCF areas when they disagree:", U[(U.ccf_area != U.location)].ccf_area.value_counts().head(6).to_dict())

**Result: not reproduced.** A layer is assigned to 97.2% of the tested units and 97.3% of the cohort (expected 87.5% and 84.9%), to 95.8 to 96.5% in VISam (96.6%) and 97.4 to 97.5% in VISrl (48.0%). The CCF-derived area matches the unit-table label for 66.0% of tested units (63.5%), within tolerance. A stricter rule outside the criterion, a layer only where the CCF structure is a visual area, gives 87.0% of tested units, 84.3% of the cohort, 96.5% in VISam and 45.5% in VISrl, all within tolerance; it suggests the paper used such a rule, which its Methods do not state. The claims below use the criterion's rule.

### Claim layer4-suppression

> Among units with an assigned layer, suppression is concentrated in layer 4; the per-session
> suppressed fraction is higher there than in the other layers in 70 of 87 sessions (80%), by a
> median of +0.116 (IQR +0.04 to +0.20; paired Wilcoxon signed-rank, p < 0.001).

In [ ]:
def l4_excess(mask, reading="a", minn=5, hl=None):
    """Per session: suppressed fraction of cohort units in layer 4 minus that in the other layers,
    (a) pooled, (b) mean of the supragranular and infragranular fractions; each fraction only with >= 5 units."""
    d = U[mask & coh & (haslayer if hl is None else hl)]
    out = []
    for k, g in d.groupby("session"):
        l4 = g[g.layer == "4"]
        if len(l4) < minn: continue
        if reading == "a":
            o = g[g.layer != "4"]
            if len(o) < minn: continue
            out.append((k, l4.supp.mean() - o.supp.mean()))
        else:
            fr = [x.supp.mean() for _, x in g[g.layer != "4"].groupby("lgroup") if len(x) >= minn]
            if len(fr) == 0: continue
            out.append((k, l4.supp.mean() - np.mean(fr)))
    return pd.DataFrame(out, columns=["session", "excess"])
def show_excess(E, label):
    p = stats.wilcoxon(E.excess).pvalue if len(E) > 1 else np.nan
    print("%s: sessions %d, layer 4 higher in %d (%.0f%%), median %+.3f (IQR %+.3f to %+.3f), Wilcoxon p = %.2g"
          % (label, len(E), (E.excess > 0).sum(), 100 * (E.excess > 0).mean(), E.excess.median(), E.excess.quantile(0.25), E.excess.quantile(0.75), p))
    return p
allm = np.ones(len(U), bool)
for r in ("a", "b"):
    show_excess(l4_excess(allm, r), "reading (%s)" % r)
print("suppressed fraction by layer (cohort units with a layer):", U[coh & haslayer].groupby("layer").supp.agg(["mean", "size"]).round(3).to_dict("index"))
# variant: layers only where the CCF structure is a visual area (claim layer-assignment-coverage)
for r in ("a", "b"):
    show_excess(l4_excess(allm, r, hl=haslayer_vis), "variant, layers only in CCF visual areas, reading (%s)" % r)


**Result: reproduced.** With all other layers pooled (reading a), the layer 4 suppressed fraction exceeds the others in 71 of 86 sessions, by a median of +0.118 (IQR +0.037 to +0.188), Wilcoxon p = 8.1e-10 (expected 70 of 87, +0.116, p < 0.001). Reading (b) gives 67 of 86, +0.115, p = 3.5e-08.

### Claim layer4-within-waveform

> The layer 4 excess of suppression persists within each waveform class (+0.137 in broad and +0.114 in
> narrow-spiking units, p < 1e-6 in both), and narrow units are not over-represented in layer 4
> (14.4% of tested layer-4 units, against 20.7% in layer 6a and 15.5% in layer 2/3).

Waveform classes from both duration readings of claim waveform-ambiguous-band.

In [ ]:
for nm in ("a", "b"):
    nar, bro = classes(nm)
    for lab, m in (("broad", bro), ("narrow", nar)):
        for r in ("a", "b"):
            show_excess(l4_excess(m, r), "duration (%s), %s units, reading (%s)" % (nm, lab, r))
    for scope, base in (("tested units", np.ones(len(U), bool)), ("cohort units", coh)):
        pct = {L: round(100 * (nar & base & (U.layer == L).values).sum() / ((nar | bro) & base & (U.layer == L).values).sum(), 1) for L in ("2/3", "4", "5", "6a")}
        print("duration (%s), narrow %% of %s with a class, by layer: %s" % (nm, scope, pct))

**Result: partially reproduced.** The layer 4 excess is positive within each waveform class: with the stored duration +0.102 in broad units (p = 6.1e-07) and +0.078 in narrow units (p = 5.7e-07), with reading (a) +0.124 (p = 1e-07) and +0.086 (p = 1.1e-05), against expected +0.137 and +0.114 at p < 1e-6; only the broad excess of reading (a) is within tolerance. With the stored duration narrow units are 15.3% of tested layer 4 units, 18.1% of layer 6a and 14.7% of layer 2/3 (expected 14.4, 20.7 and 15.5%); layer 6a is outside tolerance.

### Claim layer4-controls

> The layer 4 effect is not a sampling or detection artifact; layer 4 holds 18.8% of tested units with
> a layer against 16.6% of the cohort, its split-half detection rate matches the other layers (86.3%
> against 86.2%), and the excess survives firing-rate matching (+0.178 against +0.148 unmatched) and
> restriction to units at least 50 um from a laminar boundary (+0.176, in 63 of 72 sessions).

In [ ]:
l4 = (U.layer == "4").values
print("layer 4: %.1f%% of tested units with a layer, %.1f%% of cohort units with a layer" % (100 * l4[haslayer].mean(), 100 * l4[coh & haslayer].mean()))
# denominator (b): units testable in both contrasts as claim fig1e-cohort-size reads it under its readings (b) and (c);
# its reading (a) holds only split-half positive units, so it would make the rate 100% by construction (verifier round 1)
for nm, base in (("(a) all units with a layer", haslayer), ("(b) testable, fig1e reading (b), with a layer", haslayer & test_b.values),
                 ("(b) testable, fig1e reading (c) without the split-half gate, with a layer", haslayer & ((U.n_hit > 0) & (U.n_pas > 0) & (U.n_miss > 0)).values)):
    print("split-half detection rate %s: layer 4 %.1f%%, other layers %.1f%%" % (nm, 100 * U.sh_pos.values[base & l4].mean(), 100 * U.sh_pos.values[base & ~l4].mean()))
# rate matching across layers: octiles of log10 rate (r1) over the session's cohort units with a layer, equal counts from layer 4 and the others per octile
rng = np.random.default_rng(seed("layer4-controls"))
rm, um = [], []
for k, g in U[coh & haslayer & (U.rate_r1 > 0).values].groupby("session"):
    gl4, go = g[g.layer == "4"], g[g.layer != "4"]
    if len(gl4) < 5 or len(go) < 5: continue
    lr = np.log10(g.rate_r1.values); edges = np.quantile(lr, np.linspace(0, 1, 9)); edges[-1] += 1e-9
    o4 = np.digitize(np.log10(gl4.rate_r1.values), edges[1:-1]); oo = np.digitize(np.log10(go.rate_r1.values), edges[1:-1])
    ds = []
    for _ in range(20):
        s4, so = [], []
        for o in range(8):
            i4, io = np.where(o4 == o)[0], np.where(oo == o)[0]; nn = min(len(i4), len(io))
            if nn: s4 += list(rng.choice(i4, nn, replace=False)); so += list(rng.choice(io, nn, replace=False))
        if len(s4) >= 5: ds.append(gl4.supp.values[s4].mean() - go.supp.values[so].mean())
    if ds:
        rm.append(np.mean(ds)); um.append(gl4.supp.mean() - go.supp.mean())
rm, um = np.array(rm), np.array(um)
print("rate matched: sessions %d, median excess %+.3f (mean %+.3f), unmatched in the same sessions median %+.3f (mean %+.3f), Wilcoxon p (matched) = %.2g"
      % (len(rm), np.median(rm), rm.mean(), np.median(um), um.mean(), stats.wilcoxon(rm).pvalue))
E = l4_excess(allm, "a"); print("unmatched excess over all sessions of layer4-suppression: mean %+.3f, median %+.3f" % (E.excess.mean(), E.excess.median()))
far = (U.bdist >= 50).values
for r in ("a", "b"):
    show_excess(l4_excess(far, r), "units >= 50 um from a boundary, reading (%s)" % r)
show_excess(l4_excess((U.bdist >= 75).values, "a"), "units >= 75 um from a boundary, reading (a)")
print("boundary distance of cohort units with a layer (um): median %.0f, %% >= 50 um %.1f" % (np.nanmedian(U.bdist[coh & haslayer]), 100 * far[coh & haslayer].mean()))
show_excess(l4_excess(far, "a", hl=haslayer_vis), "variant, layers only in CCF visual areas, units >= 50 um from a boundary, reading (a)")


**Result: not reproduced.** Layer 4 holds 19.9% of tested units with a layer and 18.1% of the cohort's (expected 18.8% and 16.6%), and its split-half detection rate is 84.0% against 85.8% over all units with a layer (86.3% and 86.2%), within tolerance; over units testable in both contrasts (fig1e reading b) it is 91.1% against 92.5%, also within tolerance. The excess stays positive with rates matched (+0.125, unmatched +0.122 in the same 85 sessions; expected +0.178 and +0.148) and for units at least 50 um from a boundary (+0.142, higher in 67 of 80 sessions; expected +0.176, 63 of 72). The excesses are outside tolerance, and so is the 50 um count: 67 is within tolerance of 63 but its 80 sessions are not within 72 plus or minus 7 (reading b of the comparison gives 63 of 80). The criterion's partial clause needs the 50 um count to hold, so the claim is not reproduced.

### Claim narrow-more-suppressed-within-layer

> Narrow-spiking units are more frequently suppressed than broad-spiking units within each layer, in
> 85% of sessions (p < 0.001).

In [ ]:
for nm in ("a", "b"):
    nar, bro = classes(nm)
    d = U[coh & haslayer].assign(nar=nar[coh & haslayer], bro=bro[coh & haslayer])
    per_a, per_b = [], []
    for k, g in d.groupby("session"):
        diffs = []
        for lg, x in g.groupby("lgroup"):
            xn, xb = x[x.nar], x[x.bro]
            if len(xn) >= 5 and len(xb) >= 5:
                diffs.append(xn.supp.mean() - xb.supp.mean()); per_b.append(diffs[-1])
        if diffs: per_a.append(np.mean(diffs))
    for r, v in (("a", np.array(per_a)), ("b", np.array(per_b))):
        print("duration (%s), reading (%s): n = %d, narrow higher in %.1f%%, median difference %+.3f, Wilcoxon p = %.2g"
              % (nm, r, len(v), 100 * np.mean(v > 0), np.median(v), stats.wilcoxon(v).pvalue if len(v) > 1 else np.nan))

**Result: reproduced.** With the stored duration and one value per session averaged over layer groups (reading a), narrow units are more often suppressed than broad ones in 89.0% of 82 sessions (median +0.152, Wilcoxon p = 1.6e-11), against 85% and p < 0.001. With durations from the mean waveform, 73.9% (p = 4.6e-06).

## Population variability

The population analyses of M&M 2.6, per session and area. Whole-recording rate, the open detail (1)
of claim fig2c-cv-lower: (r1) spike count over the span from the unit's first to last spike; (r2)
spike count over the session's recorded span (0 to the last spike of any analysed unit). Units below
0.2 Hz are left out of all groups.

In [ ]:
dur_ses = U.groupby("session").t_last.max()
U["rate_r1"] = U.n_spikes / (U.t_last - U.t_first)
U["rate_r2"] = U.n_spikes / dur_ses.reindex(U.session).values
CVB = [o[2]["cvbins"] for o in OUT]
LOC = [U.index[U.session == k].values for k in range(len(OUT))]  # rows of U for session k, in its array order
def groups(k, a, rate_col):
    rows = LOC[k]; uu = U.loc[rows]
    keep = (uu[rate_col] >= 0.2).values & (uu.location == a).values
    plus = np.where(keep & uu.cohort.values)[0]; minus = np.where(keep & ~uu.cohort.values)[0]
    return plus, minus
def cv_count_matched(k, a, rate_col, rng, ndraw=20):
    plus, minus = groups(k, a, rate_col)
    if len(plus) == 0 or len(minus) < len(plus):
        return None
    X = CVB[k]
    cp = vbn.cv_windows(X[plus].sum(0, dtype=np.int64))
    draws = np.array([vbn.cv_windows(X[rng.choice(minus, len(plus), replace=False)].sum(0, dtype=np.int64)) for _ in range(ndraw)])
    call = vbn.cv_windows(X[minus].sum(0, dtype=np.int64))
    return dict(n_plus=len(plus), n_minus=len(minus), cv_plus=cp, cv_minus_t=np.nanmean(draws, 0), cv_minus_1=draws[0], cv_minus_draws=draws.astype(np.float32),
                med_plus=np.nanmedian(cp), med_minus_a=np.nanmedian(np.nanmean(draws, 0)), med_minus_b=np.nanmean(np.nanmedian(draws, 1)),
                med_minus_all=np.nanmedian(call))
def cv_rate_matched(k, a, rate_col, rng, ndraw=20):
    plus, minus = groups(k, a, rate_col)
    if len(plus) == 0 or len(minus) == 0:
        return None
    uu = U.loc[LOC[k]]; lr = np.log10(uu[rate_col].values)
    allu = np.r_[plus, minus]
    edges = np.quantile(lr[allu], np.linspace(0, 1, 9)); edges[-1] += 1e-9
    op = np.digitize(lr[plus], edges[1:-1]); om = np.digitize(lr[minus], edges[1:-1])
    X = CVB[k]; cps, cms = [], []
    nsel = sum(min((op == o).sum(), (om == o).sum()) for o in range(8))
    if nsel == 0:
        return None
    for _ in range(ndraw):
        sp, sm_ = [], []
        for o in range(8):
            ip, im = plus[op == o], minus[om == o]; nn = min(len(ip), len(im))
            if nn:
                sp += list(rng.choice(ip, nn, replace=False)); sm_ += list(rng.choice(im, nn, replace=False))
        cps.append(vbn.cv_windows(X[sp].sum(0, dtype=np.int64))); cms.append(vbn.cv_windows(X[sm_].sum(0, dtype=np.int64)))
    cps, cms = np.array(cps), np.array(cms)
    return dict(n_matched=nsel, med_plus_a=np.nanmedian(np.nanmean(cps, 0)), med_minus_a=np.nanmedian(np.nanmean(cms, 0)),
                med_plus_b=np.nanmean(np.nanmedian(cps, 1)), med_minus_b=np.nanmean(np.nanmedian(cms, 1)))
t = time.time()
CVR = {}
for rate_col in ("rate_r1", "rate_r2"):
    offs = (0, 1000, 2000, 3000, 4000) if rate_col == "rate_r1" else (0,)
    for off in offs:
        rng = np.random.default_rng(seed("fig2c-cv-lower") + off); rng2 = np.random.default_rng(seed("cv-rate-matched") + off)
        for k in range(len(OUT)):
            for a in A:
                r = cv_count_matched(k, a, rate_col, rng)
                if r is not None:
                    r2 = cv_rate_matched(k, a, rate_col, rng2)
                    CVR[(rate_col, off, k, a)] = (r, r2)
print("CV computed in %.0f s; session-area pairs with RW+ (rate r1):" % (time.time() - t), sum(1 for key in CVR if key[0] == "rate_r1" and key[1] == 0))
def cvtable(rate_col, off=0, minplus=1):
    rows = []
    for (rc, o, k, a), (r, r2) in CVR.items():
        if rc == rate_col and o == off and r["n_plus"] >= minplus:
            d = dict(session=k, area=a, n_plus=r["n_plus"], n_minus=r["n_minus"], plus=r["med_plus"], minus_a=r["med_minus_a"], minus_b=r["med_minus_b"], minus_all=r["med_minus_all"])
            if r2 is not None:
                d.update(rm_n=r2["n_matched"], rm_plus_a=r2["med_plus_a"], rm_minus_a=r2["med_minus_a"], rm_plus_b=r2["med_plus_b"], rm_minus_b=r2["med_minus_b"])
            rows.append(d)
    return pd.DataFrame(rows)

### Claim fig2c-cv-lower

> The summed activity of the reward-modulated cohort (RW+) has lower population variability than the
> remaining units (RW-) in all six areas, median CV 0.59 against 0.79, median differences -0.14 to
> -0.23 across areas, p < 0.001 in all, rank-biserial about -1.

In [ ]:
def area_tests(T, pcol, mcol):
    rows = []
    for a in A:
        d = T[T.area == a].dropna(subset=[pcol, mcol]); diff = d[pcol] - d[mcol]
        p = stats.wilcoxon(d[pcol], d[mcol]).pvalue if len(d) > 1 and (diff != 0).any() else np.nan
        rows.append(dict(area=a, sessions=len(d), median_plus=d[pcol].median(), median_minus=d[mcol].median(), median_diff=diff.median(),
                         pct_plus_lower=100 * (diff < 0).mean(), p=p, rank_biserial=vbn.rank_biserial(diff.values)))
    return pd.DataFrame(rows)
F2C = {}
for rate_col in ("rate_r1", "rate_r2"):
    for minplus in (1, 5):
        T = cvtable(rate_col, 0, minplus)
        for avg in ("a", "b"):
            res = area_tests(T, "plus", f"minus_{avg}")
            F2C[(rate_col, minplus, avg)] = (T, res)
            print("rate %s, RW+ >= %d units, draws averaged as (%s) %s: pooled median CV RW+ %.3f, RW- %.3f; pooled median difference %.3f; rank-biserial %.2f"
                  % (rate_col, minplus, avg, "CV(t)" if avg == "a" else "median CV", T.plus.median(), T[f"minus_{avg}"].median(),
                     (T.plus - T[f"minus_{avg}"]).median(), vbn.rank_biserial((T.plus - T[f"minus_{avg}"]).values)))
            print(res.round(4).to_string())
print("seed sensitivity (rate r1, RW+ >= 1, (a)): pooled median RW- CV and VISrl p under the five seeds:")
for off in (0, 1000, 2000, 3000, 4000):
    T = cvtable("rate_r1", off, 1); res = area_tests(T, "plus", "minus_a")
    print("  +%d: RW- %.4f, max p over areas %.2g, all six lower at p < 0.001: %s" % (off, T.minus_a.median(), res.p.max(), bool(((res.p < 0.001) & (res.median_diff < 0)).all())))
print("CV range (RW+, RW-): %.3f to %.3f, %.3f to %.3f" % (T.plus.min(), T.plus.max(), T.minus_a.min(), T.minus_a.max()))
print("by mouse (rate r1, RW+ >= 1): mice with RW+ median CV below RW-:", int((T.assign(subject=SEL.subject.values[T.session.values]).groupby("subject").apply(lambda d: (d.plus - d.minus_a).median() < 0)).sum()), "of", SEL.subject.nunique())

**Result: reproduced.** With RW+ of at least five units, RW+ has the lower CV in all six areas (Wilcoxon p < 0.001 in each), the pooled median CV is 0.596 in RW+ and 0.818 in RW- (expected 0.59 and 0.79), and the per-area median differences run from -0.139 to -0.247, inside -0.126 to -0.253; with at least one unit VISl's -0.265 is outside. The pooled rank-biserial is -0.84 (paper about -1). All six areas stay lower at p < 0.001 under the five seeds, and RW+ is lower in all 54 mice.

### Claim cohort-faster-firing

> The reward-modulated cohort fires faster than the rest, median whole-recording rates 5.18 Hz in RW+
> against 3.08 Hz in RW-, a ratio of 1.5 to 1.9 across the six areas.

In [ ]:
for rate_col in ("rate_r1", "rate_r2", "firing_rate"):
    for filt in (False, True):
        m = (U[rate_col] >= 0.2) if filt else np.ones(len(U), bool)
        rp, rm = U[rate_col][m & U.cohort], U[rate_col][m & ~U.cohort]
        ratios = {a: round(float(U[rate_col][m & U.cohort & (U.location == a)].median() / U[rate_col][m & ~U.cohort & (U.location == a)].median()), 2) for a in A}
        print("%s, 0.2 Hz filter %s: median RW+ %.2f Hz (n = %d), RW- %.2f Hz (n = %d); per-area ratios %s" % (rate_col, filt, rp.median(), len(rp), rm.median(), len(rm), ratios))

**Result: reproduced.** With the 0.2 Hz filter, the median whole-recording rate is 4.75 Hz in RW+ and 2.83 Hz in RW- (expected 5.18 and 3.08), and the per-area ratios are 1.49 to 1.89, inside 1.35 to 2.09. Without the filter RW- is 2.68 Hz, outside tolerance.

### Claim cv-rate-matched

> With firing-rate distributions matched as well as unit counts, RW+ stays less variable in five of
> six areas (median differences -0.02 to -0.12, p < 0.01, 69% of sessions), not in VISrl (p = 0.07);
> the pooled median difference falls from -0.19 to -0.06 and the rank-biserial from -0.81 to -0.38.

In [ ]:
for rate_col in ("rate_r1", "rate_r2"):
    for minplus in (1, 5):
        T = cvtable(rate_col, 0, minplus).dropna(subset=["rm_plus_a"])
        for avg in ("a", "b"):
            res = area_tests(T, f"rm_plus_{avg}", f"rm_minus_{avg}")
            dcm = T.plus - T[f"minus_{avg}"]; drm = T[f"rm_plus_{avg}"] - T[f"rm_minus_{avg}"]
            print("rate %s, RW+ >= %d, (%s): pairs %d; pooled median difference count-matched %.4f, rate-matched %.4f; rank-biserial %.2f -> %.2f; %% of pairs RW+ lower (rate-matched) %.1f"
                  % (rate_col, minplus, avg, len(T), dcm.median(), drm.median(), vbn.rank_biserial(dcm.values), vbn.rank_biserial(drm.values), 100 * (drm < 0).mean()))
            print(res.round(4).to_string())
for off in (0, 1000, 2000, 3000, 4000):
    T = cvtable("rate_r1", off, 1).dropna(subset=["rm_plus_a"]); drm = T.rm_plus_a - T.rm_minus_a
    print("  seed +%d: pooled rate-matched difference %.4f, rank-biserial %.3f" % (off, drm.median(), vbn.rank_biserial(drm.values)))

**Result: partially reproduced.** Matching rates as well as counts attenuates the difference, pooled from -0.194 to -0.048, with the rank-biserial from -0.80 to -0.38 (expected -0.19 to -0.06, -0.81 to -0.38) and RW+ lower in 64.6% of pairs (69%). RW+ stays lower at p < 0.01 in VISl, VISal and VISam, not in VISp (p = 0.025) or VISpm (p = 0.117), and VISrl has p = 0.197 (paper 0.07). The rate-matched pooled difference of -0.048 is outside tolerance (-0.066 to -0.054), and no reading gives p < 0.01 in VISp.

### Claim cv-unmatched-reverses

> Without matching the unit count, the sign of the RW+ against RW- CV comparison reverses, because
> RW+ is always the smaller group and summed variability falls with population size.

In [ ]:
for rate_col in ("rate_r1", "rate_r2"):
    T = cvtable(rate_col, 0, 1)
    allp = []
    for k in range(len(OUT)):
        for a in A:
            p, mi = groups(k, a, rate_col)
            if len(p):
                allp.append((len(p), len(mi)))
    allp = np.array(allp)
    print("rate %s: session-area pairs with RW+ %d; RW+ smaller than RW- in %d; pooled median of RW+ minus all-RW- CV %.3f (RW+ %.3f, RW- %.3f); %% pairs with RW+ higher %.1f"
          % (rate_col, len(allp), (allp[:, 0] < allp[:, 1]).sum(), (T.plus - T.minus_all).median(), T.plus.median(), T.minus_all.median(), 100 * ((T.plus - T.minus_all) > 0).mean()))

**Result: partially reproduced.** Without count matching the pooled median of RW+ minus RW- CV is positive (+0.041), so the sign reverses, but RW+ is the smaller group in 555 of 576 session and area pairs, not in all.

### Claim fig2d-cv-correlations

> The CV(t) series co-fluctuate most within RW- across areas (mean correlation +0.58), less between
> RW+ and RW- (+0.45) and least within RW+ (+0.38); all 66 pairwise correlations exceed an
> autocorrelation-preserving null (mean null correlation at most 0.014, z at least 15) and are
> significant after FDR (largest p_FDR 1.0e-12).

Series: RW+ CV(t) and the count-matched RW- CV(t) (20 draws averaged), rate r1, RW+ >= 1 unit.

In [ ]:
COMB = [(a, g) for g in ("+", "-") for a in A]
PAIRS = [(i, j) for i in range(12) for j in range(i + 1, 12)]
def block_of(i, j):
    (ai, gi), (aj, gj) = COMB[i], COMB[j]
    if gi == gj == "-": return "within RW-"
    if gi == gj == "+": return "within RW+"
    return "cross, same area" if ai == aj else "cross, different areas"
SER = []
for k in range(len(OUT)):
    X = {}
    for a in A:
        key = ("rate_r1", 0, k, a)
        if key in CVR:
            X[(a, "+")] = CVR[key][0]["cv_plus"]; X[(a, "-")] = CVR[key][0]["cv_minus_t"]
    SER.append(X)
def ses_corr(X, shift=None, rng=None, block=None):
    R = np.full((12, 12), np.nan)
    have = [i for i, c in enumerate(COMB) if c in X]
    if len(have) < 2: return R
    M = np.array([X[COMB[i]] for i in have])
    if shift == "circ":
        M = np.array([np.roll(m, rng.integers(len(m))) for m in M])
    elif shift == "block":
        nb_ = int(np.ceil(M.shape[1] / 30)); M = np.array([np.concatenate([m[b * 30:(b + 1) * 30] for b in rng.permutation(nb_)]) for m in M])
    ok = np.all(np.isfinite(M), 0)
    C = np.corrcoef(M[:, ok])
    R[np.ix_(have, have)] = C
    return R
RS = np.array([ses_corr(X) for X in SER])
mean_r = np.nanmean(RS, 0)
n_ses = np.sum(np.isfinite(RS), 0)
# reading (b): concatenated series per pair
from statsmodels.stats.multitest import multipletests
rows = []
for (i, j) in PAIRS:
    xs, ys = [], []
    for X in SER:
        if COMB[i] in X and COMB[j] in X:
            ok = np.isfinite(X[COMB[i]]) & np.isfinite(X[COMB[j]]); xs.append(X[COMB[i]][ok]); ys.append(X[COMB[j]][ok])
    x, y = np.concatenate(xs), np.concatenate(ys)
    r, p = stats.pearsonr(x, y)
    rows.append(dict(pair=f"{COMB[i][0]}{COMB[i][1]}~{COMB[j][0]}{COMB[j][1]}", block=block_of(i, j), sessions=int(n_ses[i, j]), mean_r=mean_r[i, j], concat_r=r, concat_p=p, n=len(x)))
F2D = pd.DataFrame(rows)
F2D["p_fdr"] = multipletests(F2D.concat_p, method="fdr_bh")[1]
t = time.time()
for kind, sd in (("circ", seed("fig2d-cv-correlations")), ("block", seed("fig2d-cv-correlations") + 1)):
    rng = np.random.default_rng(sd)
    NUL2 = np.array([np.nanmean(np.array([ses_corr(X, kind, rng) for X in SER]), 0) for _ in range(1000)])
    F2D[f"null_mean_{kind}"] = [NUL2[:, i, j].mean() for i, j in PAIRS]
    F2D[f"z_{kind}"] = [(mean_r[i, j] - NUL2[:, i, j].mean()) / NUL2[:, i, j].std() for i, j in PAIRS]
print("nulls computed in %.0f s" % (time.time() - t))
print(F2D.groupby("block")[["mean_r", "concat_r", "null_mean_circ", "null_mean_block", "z_circ", "z_block"]].agg(["mean", "min", "max"]).round(3).T)
cross_all = F2D.block.str.startswith("cross")
for nm, m in (("within RW-", F2D.block == "within RW-"), ("cross, all 36", cross_all), ("cross, different areas", F2D.block == "cross, different areas"), ("within RW+", F2D.block == "within RW+")):
    print("%s: mean of per-session means %.3f, concatenated %.3f" % (nm, F2D.mean_r[m].mean(), F2D.concat_r[m].mean()))
# variant (verifier round 1): RW- as a single count-matched draw, so that its CV(t) is not smoothed by averaging 20 draws
SER1 = []
for k in range(len(OUT)):
    X = {}
    for a in A:
        key = ("rate_r1", 0, k, a)
        if key in CVR:
            X[(a, "+")] = CVR[key][0]["cv_plus"]; X[(a, "-")] = CVR[key][0]["cv_minus_1"]
    SER1.append(X)
RS1 = np.array([ses_corr(X) for X in SER1]); mean_r1 = np.nanmean(RS1, 0)
F2D["mean_r_single"] = [mean_r1[i, j] for i, j in PAIRS]
print("single RW- draw: mean of per-session means by block", F2D.groupby("block").mean_r_single.mean().round(3).to_dict(), "| cross, all 36: %.3f" % F2D.mean_r_single[cross_all].mean())
# variant (verifier round 2): correlations formed on each of the 20 RW- draws, then averaged over the draws
RSD = np.zeros((len(SER), 12, 12)); NSD = np.zeros((len(SER), 12, 12))
for d in range(20):
    SERd = []
    for k in range(len(OUT)):
        X = {}
        for a in A:
            key = ("rate_r1", 0, k, a)
            if key in CVR:
                X[(a, "+")] = CVR[key][0]["cv_plus"]; X[(a, "-")] = CVR[key][0]["cv_minus_draws"][d].astype(float)
        SERd.append(X)
    R_ = np.array([ses_corr(X) for X in SERd]); okR = np.isfinite(R_)
    RSD += np.where(okR, R_, 0); NSD += okR
RSD = np.where(NSD > 0, RSD / np.maximum(NSD, 1), np.nan); mean_rd = np.nanmean(RSD, 0)
F2D["mean_r_perdraw"] = [mean_rd[i, j] for i, j in PAIRS]
print("per-draw r averaged over the 20 draws: mean of per-session means by block", F2D.groupby("block").mean_r_perdraw.mean().round(3).to_dict(), "| cross, all 36: %.3f" % F2D.mean_r_perdraw[cross_all].mean())
print("pairs with z >= 15: circular %d, block %d of 66; largest null mean %.4f; largest p_FDR %.2g; min sessions per pair %d"
      % ((F2D.z_circ >= 15).sum(), (F2D.z_block >= 15).sum(), F2D[["null_mean_circ", "null_mean_block"]].abs().max().max(), F2D.p_fdr.max(), F2D.sessions.min()))

**Result: partially reproduced.** Averaged over sessions, the mean correlation is 0.523 within RW-, 0.416 across RW+ and RW- (36 pairs) and 0.354 within RW+, all within tolerance and in the paper's order. Every pair is above both nulls (null means at most 0.027), but 4 of 66 pairs have z below 15 against the circular-shift null (minimum 13.8); all 66 reach z >= 15 against the block null. The largest FDR-corrected p of the concatenated series is 1.4e-113. RW- here is the mean CV(t) of 20 count-matched draws, as claim fig2c-cv-lower reading (a) has it. With a single draw, so that RW- is as noisy as RW+, the means are 0.300 within RW-, 0.318 across and 0.354 within RW+, and the order is lost: the RW- excess depends on the averaging. Forming the correlations on each of the 20 draws and averaging them over the draws (a second variant) gives 0.304 within RW-, 0.322 across and 0.354 within RW+, also out of the paper's order. The outcome rests on correlating the averaged series, which Materials and methods 2.6 describes (RW- "repeatedly sampled without replacement (20 draws, averaged)", then the curves averaged across sessions); this is an assumption.

### Claim cv-correlation-block-excess

> Within sessions, the RW- block's mean CV(t) correlation exceeds the RW+ block's by a median of
> +0.21, in 87% of sessions (p < 1e-13).

In [ ]:
wm = [(i, j) for (i, j) in PAIRS if block_of(i, j) == "within RW-"]; wp = [(i, j) for (i, j) in PAIRS if block_of(i, j) == "within RW+"]
ex = []
for k, R in enumerate(RS):
    a = np.nanmean([R[i, j] for i, j in wm]) if np.isfinite([R[i, j] for i, j in wm]).any() else np.nan
    b = np.nanmean([R[i, j] for i, j in wp]) if np.isfinite([R[i, j] for i, j in wp]).any() else np.nan
    ex.append(a - b)
ex = np.array(ex); ex = ex[np.isfinite(ex)]
ex1 = []
for R in RS1:
    a = np.nanmean([R[i, j] for i, j in wm]) if np.isfinite([R[i, j] for i, j in wm]).any() else np.nan
    b = np.nanmean([R[i, j] for i, j in wp]) if np.isfinite([R[i, j] for i, j in wp]).any() else np.nan
    ex1.append(a - b)
ex1 = np.array(ex1); ex1 = ex1[np.isfinite(ex1)]
exd = []
for R in RSD:
    a = np.nanmean([R[i, j] for i, j in wm]) if np.isfinite([R[i, j] for i, j in wm]).any() else np.nan
    b = np.nanmean([R[i, j] for i, j in wp]) if np.isfinite([R[i, j] for i, j in wp]).any() else np.nan
    exd.append(a - b)
exd = np.array(exd); exd = exd[np.isfinite(exd)]
print("variant, per-draw r averaged over the 20 draws: sessions %d; median excess %.4f; %% positive %.1f; Wilcoxon p = %.3g" % (len(exd), np.median(exd), 100 * np.mean(exd > 0), stats.wilcoxon(exd).pvalue))
print("variant, single RW- draw: sessions %d; median excess %.4f; %% positive %.1f; Wilcoxon p = %.3g" % (len(ex1), np.median(ex1), 100 * np.mean(ex1 > 0), stats.wilcoxon(ex1).pvalue))
print("sessions with at least two areas with both groups: %d; median excess %.4f; %% sessions positive %.1f; Wilcoxon p = %.3g"
      % (len(ex), np.median(ex), 100 * np.mean(ex > 0), stats.wilcoxon(ex).pvalue))

**Result: partially reproduced.** Within 100 sessions, the RW- block's mean correlation exceeds the RW+ block's in 83.0% (expected 87%) by a median of +0.158 (expected +0.21, outside tolerance), Wilcoxon p = 1.8e-12 (expected p < 1e-13). With a single RW- draw in place of the mean of 20 (a variant), the median excess is -0.050, positive in 43.0% of sessions (Wilcoxon p = 0.036): the excess comes from averaging the RW- draws. With the correlations formed on each draw and averaged over the 20 draws, the median excess is -0.038, positive in 39.0% of sessions (p = 0.014). The outcome rests on correlating the averaged series, an assumption.

### Claim fig2f-pre-post-cv

> Population CV rises from pre- to post-stimulus (image change) windows. In RW- the rise is
> significant in all six areas in both blocks (p < 0.001); in RW+ it is significant in the active
> block in VISp, VISl, VISpm and VISrl only, and in passive replay in VISp, VISl and VISal only. In
> VISp the RW+ median CV rises from 0.74 to 0.95 in the active block and from 0.69 to 0.73 in replay.

Counts in 50 ms bins from -0.75 to +0.75 s around each image change are stored per unit. Readings as
in the criterion: windows (a) -0.5 to 0 and 0 to 0.75 s, (b) -0.75 to 0 and 0 to 0.75 s; events (a)
all image changes, (b) in the active block hit and miss changes only; CV (a) per event window, median
over events, (b) over all bins of the window pooled over events. RW- count-matched to RW+ (20 draws,
averaged), rate r1, RW+ >= 1 unit.

In [ ]:
PAPER_SIG = {("active", a): a in ("VISp", "VISl", "VISpm", "VISrl") for a in A}
PAPER_SIG.update({("replay", a): a in ("VISp", "VISl", "VISal") for a in A})
WINS = {"a": ((5, 15), (15, 30)), "b": ((0, 15), (15, 30))}
def cv_ev(P, w, how):
    x = P[:, w[0]:w[1]].astype(float)
    if how == "a":
        m = x.mean(1); s = x.std(1)
        with np.errstate(all="ignore"):
            return np.nanmedian(np.where(m > 0, s / m, np.nan))
    v = x.ravel(); return v.std() / v.mean() if v.mean() > 0 else np.nan
rng = np.random.default_rng(seed("fig2f-pre-post-cv"))
rec = []
for k, (meta, df, arr, ev) in enumerate(OUT):
    for a in A:
        plus, minus = groups(k, a, "rate_r1")
        if len(plus) == 0 or len(minus) < len(plus): continue
        draws = [rng.choice(minus, len(plus), replace=False) for _ in range(20)]
        for blk, key in (("active", "pp_act"), ("replay", "pp_pas")):
            PPk = arr[key]
            for evr in ("a", "b"):
                sel = np.ones(PPk.shape[1], bool) if (evr == "a" or blk == "replay") else arr["act_hitmiss"]
                Pp = PPk[plus][:, sel].sum(0, dtype=np.int64)
                Pm = [PPk[d][:, sel].sum(0, dtype=np.int64) for d in draws]
                for wr in ("a", "b"):
                    for how in ("a", "b"):
                        w0, w1 = WINS[wr]
                        rec.append(dict(session=k, area=a, block=blk, events=evr, windows=wr, cv=how,
                                        plus_pre=cv_ev(Pp, w0, how), plus_post=cv_ev(Pp, w1, how),
                                        minus_pre=np.nanmean([cv_ev(x, w0, how) for x in Pm]), minus_post=np.nanmean([cv_ev(x, w1, how) for x in Pm])))
F2F = pd.DataFrame(rec)
rows = []
for (evr, wr, how), d in F2F.groupby(["events", "windows", "cv"]):
    r = dict(events=evr, windows=wr, cv=how); nmatch = 0; minus_ok = 0; plus_sig = 0
    for blk in ("active", "replay"):
        for a in A:
            x = d[(d.block == blk) & (d.area == a)].dropna()
            pp_ = stats.wilcoxon(x.plus_pre, x.plus_post).pvalue; pm_ = stats.wilcoxon(x.minus_pre, x.minus_post).pvalue
            sig = (pp_ < 0.05) and (x.plus_post.median() > x.plus_pre.median())
            nmatch += sig == PAPER_SIG[(blk, a)]; plus_sig += sig
            minus_ok += (pm_ < 0.001) and (x.minus_post.median() > x.minus_pre.median())
    vp = d[(d.area == "VISp")]
    r.update(minus_cells_rise_p001=minus_ok, plus_cells_sig=plus_sig, plus_cells_matching_paper=nmatch,
             VISp_act_pre=vp[vp.block == "active"].plus_pre.median(), VISp_act_post=vp[vp.block == "active"].plus_post.median(),
             VISp_rep_pre=vp[vp.block == "replay"].plus_pre.median(), VISp_rep_post=vp[vp.block == "replay"].plus_post.median())
    rows.append(r)
F2Fs = pd.DataFrame(rows); print(F2Fs.round(3).to_string())

In [ ]:
# per-cell p values for the reading combination with the most RW+ cells matching
best = F2Fs.sort_values(["plus_cells_matching_paper", "minus_cells_rise_p001"], ascending=False).iloc[0]
d = F2F[(F2F.events == best.events) & (F2F.windows == best.windows) & (F2F.cv == best.cv)]
rows = []
for blk in ("active", "replay"):
    for a in A:
        x = d[(d.block == blk) & (d.area == a)].dropna()
        rows.append(dict(block=blk, area=a, sessions=len(x), plus_pre=x.plus_pre.median(), plus_post=x.plus_post.median(), plus_p=stats.wilcoxon(x.plus_pre, x.plus_post).pvalue,
                         paper_plus_sig=PAPER_SIG[(blk, a)], minus_pre=x.minus_pre.median(), minus_post=x.minus_post.median(), minus_p=stats.wilcoxon(x.minus_pre, x.minus_post).pvalue))
print("events (%s), windows (%s), CV (%s)" % (best.events, best.windows, best.cv)); print(pd.DataFrame(rows).round(4).to_string())

**Result: not reproduced.** The CV rises from pre to post in RW- in all 12 area and block cells at p < 0.001 under six of the eight reading combinations, but RW+ rises significantly in all 12 cells as well, under every combination, so 7 of 12 cells match the paper's RW+ pattern (at least 10 needed). In VISp the RW+ median CV goes from 0.603 to 0.698 (pre) to 0.882 to 0.914 (post) in the active block and from 0.541 to 0.635 to 0.663 to 0.726 in replay across the combinations (expected 0.74 to 0.95 and 0.69 to 0.73); all four are within tolerance under windows (b) with pooled CV (b).

## Temporal profiles

### Claim fig3a-archetypes

> NMF of the cohort's standardized peri-reward profiles gives six signed archetypes, three
> anticipatory with peaks between -445 and -175 ms, one peaking near reward (-45 ms), one late (+1405
> ms) and one broad and slow.

Profiles as M&M 2.7: -0.5 to +1.5 s in 10 ms bins with 100 ms padding, smoothed (sigma 25 ms), padding
removed, z-scored within unit; channels [max(X, 0), max(-X, 0)]; scikit-learn NMF rank 6, coordinate
descent, tol 1e-4, max_iter 600. Initialisation readings (a) scikit-learn default, (b) random, seeded
with the claim's seed.

In [ ]:
from sklearn.decomposition import NMF, PCA
import itertools
i0 = int(round((-0.6 - vbn.WIDE[0]) / vbn.BIN)); i1 = int(round((1.6 - vbn.WIDE[0]) / vbn.BIN))
prof = vbn.smooth(RATE[coh][:, i0:i1])[:, 10:-10]
sd_ = prof.std(1); keep_p = sd_ > 0
Z = (prof[keep_p] - prof[keep_p].mean(1, keepdims=True)) / sd_[keep_p, None]
print("response matrix:", Z.shape, "| cohort units with a flat profile (dropped):", (~keep_p).sum())
XN = np.hstack([np.maximum(Z, 0), np.maximum(-Z, 0)])
tc = -0.5 + vbn.BIN * (np.arange(200) + 0.5)  # bin centres (s)
def roles_ok(peaks):
    for perm in itertools.permutations(range(6)):
        a1, a2, a3, nr, lt, _ = [peaks[i] for i in perm]
        if all(-0.645 <= x <= 0.025 for x in (a1, a2, a3)) and abs(nr + 0.045) <= 0.2 and abs(lt - 1.405) <= 0.2:
            return True
    return False
FITS = {}
def ve(X, Xh):
    return 1 - ((X - Xh) ** 2).sum() / (X ** 2).sum()
t = time.time()
for init in ("default", "random"):
    kw = dict(n_components=6, solver="cd", tol=1e-4, max_iter=600, random_state=seed("fig3a-archetypes"))
    if init == "random": kw["init"] = "random"
    nmf = NMF(**kw); Wm = nmf.fit_transform(XN); H = nmf.components_
    Hs = H[:, :200] - H[:, 200:]
    pk_a = tc[np.argmax(Hs, 1)]; pk_b = tc[np.argmax(np.abs(Hs), 1)]
    order = np.argsort(pk_a)
    FITS[init] = dict(W=Wm, H=H, Hs=Hs, pk_a=pk_a, pk_b=pk_b, n_iter=nmf.n_iter_, err=nmf.reconstruction_err_)
    print("init %s: %d iterations; peaks (max of signed archetype, ms): %s; peaks (max |.|, ms): %s; roles fit: (a) %s, (b) %s"
          % (init, nmf.n_iter_, np.round(np.sort(pk_a) * 1e3).astype(int).tolist(), np.round(pk_b[order] * 1e3).astype(int).tolist(), roles_ok(pk_a), roles_ok(pk_b)))
print("NMF in %.0f s" % (time.time() - t))
print("random initialisation under the claim's seed plus 1000 to 4000:")
for off in (1000, 2000, 3000, 4000):
    nm_ = NMF(n_components=6, solver="cd", tol=1e-4, max_iter=600, init="random", random_state=seed("fig3a-archetypes") + off)
    W2 = nm_.fit_transform(XN); H2 = nm_.components_; Hs2 = H2[:, :200] - H2[:, 200:]
    en2, P2 = vbn.effective_number(W2)
    print("  +%d: peaks (max |.|, ms) %s, roles fit (a) %s, (b) %s; median effective number %.3f; %% max weight >= 0.9 %.3f; VE %.3f"
          % (off, np.round(np.sort(tc[np.argmax(np.abs(Hs2), 1)]) * 1e3).astype(int).tolist(), roles_ok(tc[np.argmax(Hs2, 1)]), roles_ok(tc[np.argmax(np.abs(Hs2), 1)]),
             np.nanmedian(en2), 100 * np.nanmean(np.nanmax(P2, 1) >= 0.9), ve(Z, W2 @ Hs2) if "ve" in globals() else np.nan))

**Result: reproduced, resting on an assumption.** The response matrix is 10,822 by 200. Taking the peak as the maximum absolute value (reading b), the six archetypes peak at -445, -285 and -185 ms (anticipatory), -35 ms, +1385 ms and -495 ms with the default initialisation, which fits the paper's roles; with random initialisation the peaks are -445, -285, -185, +185, -495 and +1395 ms, which also fit. With the peak as the maximum of the signed archetype (reading a) no role assignment fits.

### Claim fig3b-mixtures

> Units are mixtures of archetypes, with a median effective number of 2.72 and only 1.0% dominated by
> one archetype (weight at least 0.9); the soft mixture explains about twice the variance of a hard
> single-archetype assignment (0.72 against 0.36).

In [ ]:
from sklearn.mixture import GaussianMixture
def ve(X, Xh):
    return 1 - ((X - Xh) ** 2).sum() / (X ** 2).sum()
for init in ("default", "random"):
    Wm, H, Hs = FITS[init]["W"], FITS[init]["H"], FITS[init]["Hs"]
    en_a, P_a = vbn.effective_number(Wm)
    en_b, P_b = vbn.effective_number(Wm * np.linalg.norm(H, axis=1)[None, :])
    FITS[init]["P_a"], FITS[init]["P_b"] = P_a, P_b
    ve_nmf = ve(Z, Wm @ Hs)
    pcs = PCA(6, random_state=seed("fig3b-mixtures")).fit_transform(Z)
    lab = GaussianMixture(6, covariance_type="full", random_state=seed("fig3b-mixtures")).fit_predict(pcs)
    cent = np.array([Z[lab == c].mean(0) for c in range(6)])
    hard_a = cent[lab]
    sc = (Z * hard_a).sum(1) / (hard_a * hard_a).sum(1); hard_b = hard_a * sc[:, None]
    pca_rec = PCA(6).fit(Z); ve_pca = ve(Z, pca_rec.inverse_transform(pca_rec.transform(Z)))
    print("init %s: median effective number (a) %.3f, (b) %.3f; %% with max weight >= 0.9 (a) %.3f, (b) %.3f; VE NMF %.3f; hard (a) %.3f, (b) %.3f; rank-6 PCA %.3f"
          % (init, np.nanmedian(en_a), np.nanmedian(en_b), 100 * np.nanmean(np.nanmax(P_a, 1) >= 0.9), 100 * np.nanmean(np.nanmax(P_b, 1) >= 0.9), ve_nmf, ve(Z, hard_a), ve(Z, hard_b), ve_pca))

**Result: not reproduced.** The median effective number of archetypes is 2.755 (reading a, default initialisation; expected 2.72) and the NMF mixture explains 0.730 of the variance (0.72), within tolerance; 0.887% of units have a weight >= 0.9 (expected 1.0%, tolerance from 0.9%), and the hard single-archetype assignment explains 0.529 to 0.551 (expected 0.36), so the mixture explains 1.33 to 1.38 times as much, not twice.

### Claim fig3c-area-overlap

> The six areas' distributions in the archetype simplex overlap almost entirely; the between-area
> variance is small (eta squared 0.004) though above a label-shuffle null (p < 0.001).

In [ ]:
area_p = U.location.values[coh][keep_p]
def eta2(P, lab):
    P = np.nan_to_num(P); mu = P.mean(0)
    tot = ((P - mu) ** 2).sum()
    bet = sum((lab == a).sum() * ((P[lab == a].mean(0) - mu) ** 2).sum() for a in A)
    return bet / tot
for init in ("default", "random"):
    for nm in ("P_a", "P_b"):
        P = FITS[init][nm]; e = eta2(P, area_p)
        rng = np.random.default_rng(seed("fig3c-area-overlap"))
        nul = np.array([eta2(P, rng.permutation(area_p)) for _ in range(1000)])
        print("init %s, weights %s: eta squared %.4f; shuffle null mean %.5f, max %.5f; p = %.4f" % (init, nm[-1], e, nul.mean(), nul.max(), (1 + (nul >= e).sum()) / 1001))

**Result: not reproduced.** Eta squared of simplex position by area is 0.0114 to 0.0212 across the readings (expected 0.004, and below 0.01 for a partial result); no shuffle of 1,000 reaches it (null maximum 0.0012).

### Claim fig3d-anticipatory-gradient

> The fraction of cohort units with an anticipatory profile is highest in VISp (89%) and VISl (77%)
> and lower and similar across the four higher-order areas (66 to 69%).

In [ ]:
def mz(lo, hi):
    m = (tc >= lo) & (tc < hi); return Z[:, m].mean(1)
ANT = {"a": mz(-0.5, 0) > mz(0, 1.5), "b": mz(-0.5, 0) > mz(0, 0.5)}
for init in ("default", "random"):
    pk = FITS[init]["pk_a"]; antic = np.argsort(pk)[:3]  # the three archetypes with the earliest peaks
    W_ = FITS[init]["W"]
    ANT[f"c_{init}"] = W_[:, antic].sum(1) > np.delete(W_, antic, axis=1).sum(1)
rows = []
for nm, lab in ANT.items():
    r = {"reading": nm}
    for a in A: r[a] = 100 * lab[area_p == a].mean()
    rows.append(r)
F3D = pd.DataFrame(rows); print(F3D.round(1).to_string())

**Result: reproduced, resting on an assumption.** Reading (a), the mean standardized response over -0.5 to 0 s against 0 to +1.5 s, gives anticipatory fractions of 91.4% in VISp and 78.6% in VISl (expected 89% and 77%) and 68.8 to 72.8% in the other four areas, inside 59.4 to 75.9%. Reading (b) puts VISpm at 76.1%, and the archetype-weight reading (c) gives 37.5 to 82.5%.

### Claim anticipatory-direction-overlap

> The anticipatory label largely re-expresses the suppression label; they agree for 80.7% of the
> cohort (phi = 0.56), 91.5% of suppressed against 39.9% of enhanced units carry an anticipatory
> ramp, and standardizing areas to the overall direction composition leaves 31% of the VISp to VISam
> gradient (+0.07 of +0.23).

In [ ]:
sup_p = U.supp.values[coh][keep_p]
for nm, lab in ANT.items():
    agree = np.mean(lab == sup_p)
    phi = np.corrcoef(lab.astype(float), sup_p.astype(float))[0, 1]
    ws, we = sup_p.mean(), 1 - sup_p.mean()
    frac = {a: lab[area_p == a].mean() for a in ("VISp", "VISam")}
    std_ = {a: ws * lab[(area_p == a) & sup_p].mean() + we * lab[(area_p == a) & ~sup_p].mean() for a in ("VISp", "VISam")}
    std_pp = {a: 0.655 * lab[(area_p == a) & sup_p].mean() + 0.345 * lab[(area_p == a) & ~sup_p].mean() for a in ("VISp", "VISam")}
    print("reading %s: agreement %.1f%%, phi %.3f; suppressed anticipatory %.1f%%, enhanced %.1f%%; VISp - VISam %.3f; standardized (this cohort's %.3f/%.3f) %.3f, (paper's 0.655/0.345) %.3f"
          % (nm, 100 * agree, phi, 100 * lab[sup_p].mean(), 100 * lab[~sup_p].mean(), frac["VISp"] - frac["VISam"], ws, we, std_["VISp"] - std_["VISam"], std_pp["VISp"] - std_pp["VISam"]))

**Result: partially reproduced, resting on an assumption.** Under reading (a), the anticipatory and suppressed labels agree for 84.2% of the cohort, 94.7% of suppressed and 38.3% of enhanced units are anticipatory, VISp exceeds VISam by 0.227, and standardizing to the cohort's composition leaves 0.066, all within tolerance (expected 80.7%, 91.5%, 39.9%, 0.23 and 0.07); phi is 0.622, just outside 0.504 to 0.616.

## Licking controls

### Claim hits-vs-false-alarms

> In the underpowered hits against false alarms contrast (486 false alarms in 102 sessions, a median
> of 4 per session against 160 hits), 14.6% (VISp) to 22.2% (VISpm) of cohort units differ after FDR
> correction, with a median SI of -0.56 in VISp, deeper suppression when the lick is rewarded.

Readings: false alarms aligned (a) to the first lick after the catch (sham) change, (b) to the sham
change plus the session median latency; baseline 0.5 s before the sham change; with and without the
responsiveness filter; BH per area over cohort units.

In [ ]:
nfa = np.array([e["n_fa_trials"] for e in EV]); nfa_t = np.array([len(e["fa_change"]) for e in EV]); nh = np.array([e["n_hit_trials"] for e in EV])
print("false-alarm trials without a sham-change presentation, left out: %d" % sum(e["n_fa_no_sham"] for e in EV))
print("false-alarm trials %d (with a sham change time %d) in %d sessions with any; median per session %.1f (over sessions with any %.1f); median hits per session %.1f"
      % (nfa.sum(), nfa_t.sum(), (nfa > 0).sum(), np.median(nfa), np.median(nfa[nfa > 0]), np.median(nh)))
for k in ("FAa", "FAb"):
    for resp in (True, False):
        r_ = ((U[f"{k}_p1"] < 0.05) | (U[f"{k}_p2"] < 0.05)).values if resp else np.ones(len(U), bool)
        elig = coh & r_ & np.isfinite(U[f"{k}_pmw"].values)
        sig = vbn.bh_by(U, f"{k}_pmw", "location", elig)
        fr = {a: round(100 * sig[coh & (U.location == a).values].mean(), 1) for a in A}
        vp = coh & (U.location == "VISp").values
        print("%s, responsiveness filter %s: %% of cohort significant %s; median SI VISp (all cohort) %.3f, (significant) %.3f"
              % (k, resp, fr, np.nanmedian(U[f"{k}_si"][vp]), np.nanmedian(U[f"{k}_si"][vp & sig])))

**Result: not reproduced.** There are 516 false-alarm trials in 102 sessions (expected 486 in 102) and a median of 152.5 hits per session (160), within tolerance, but a median of 4.5 false alarms per session (4). Only 2.0 to 9.9% of cohort units per area differ from hits after correction under the four readings (expected 14.6 to 22.2%), and the median SI in VISp is -0.671 or -0.766 (expected -0.56). Every false-alarm trial has a sham-change presentation, so all 516 times are on the clock of the hit trials' change presentations (none was left out).

### Claim hits-vs-lick-bouts

> Against the onsets of unrewarded lick bouts (a median of 415 per session), 74.9% (VISp) to 88.8%
> (VISrl) of cohort units differ from the hit lick after correction, with deeper suppression on
> rewarded licks in VISp (median SI -0.50).

Bouts: licks of the active block split at inter-lick gaps above 250 or 500 ms (an assumption); a bout
is unrewarded when no reward falls between its first and last lick. The hit lick is the first lick
after the change of a hit trial. Baselines (a) 0.5 s before each event; (b) for hits 0.5 s before the
change. Responsiveness filter applied as in Families A and B; BH per area over cohort units.

In [ ]:
for gap in (250, 500):
    nb_ = np.array([len(e[f"bout_on_{gap}"]) for e in EV])
    print("gap %d ms: unrewarded bouts per session median %.0f (min %d, max %d)" % (gap, np.median(nb_), nb_.min(), nb_.max()))
    for bl in ("a", "b"):
        k = f"LB{gap}{bl}"
        r_ = ((U[f"{k}_p1"] < 0.05) | (U[f"{k}_p2"] < 0.05)).values
        for resp in (True, False):
            elig = coh & (r_ if resp else True) & np.isfinite(U[f"{k}_pmw"].values)
            sig = vbn.bh_by(U, f"{k}_pmw", "location", elig)
            fr = {a: round(100 * sig[coh & (U.location == a).values].mean(), 1) for a in A}
            print("  baseline (%s), responsiveness filter %s: %% of cohort significant %s; median SI VISp %.3f" % (bl, resp, fr, np.nanmedian(U[f"{k}_si"][coh & (U.location == "VISp").values])))

**Result: partially reproduced, resting on an assumption.** With bouts split at 500 ms gaps, there are a median of 438 unrewarded bouts per session (expected 415); with the hit baseline before the change (b), 71.0% (VISp) to 87.8% (VISrl) of cohort units differ, every area inside 67.4 to 97.7%, but the median SI in VISp is -0.132 (expected -0.50). With the baseline before each lick (a), the VISp SI is -0.785 and VISp's 91.9% is outside the expected range.

In [ ]:
print("notebook run time %.1f min; peak memory of this process %.1f GB" % ((time.time() - T0) / 60, resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6))

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and
the differences from the paper's method.

| Claim | Outcome |
|---|---|
| data-sessions-units | partially reproduced (assumption) |
| blocks-no-reward-in-replay | reproduced |
| reward-interval-min | reproduced |
| latency-change-to-reward | partially reproduced |
| lick-reward-coincide | partially reproduced |
| fig1e-cohort-size | partially reproduced (assumption) |
| fig1d-split-half-above-null | reproduced (assumption) |
| fig1c-example-cells | reproduced |
| fig1e-suppressed-fraction-by-area | reproduced |
| fig1l-session-gradient | partially reproduced |
| latency-jitter-control | reproduced |
| waveform-ambiguous-band | not reproduced |
| fig1jk-narrow-fraction-by-area | reproduced |
| fig1h-median-isi-narrow-broad | reproduced |
| fig1gi-waveform-separation | reproduced |
| narrow-more-suppressed | reproduced |
| narrow-more-suppressed-within-layer | reproduced (outside input) |
| fig1mn-modulation-index-negative | partially reproduced |
| fig1m-narrow-stronger-m | reproduced |
| m-baseline-sensitivity | partially reproduced |
| layer-assignment-coverage | not reproduced (outside input) |
| layer4-suppression | reproduced (outside input) |
| layer4-within-waveform | partially reproduced (outside input) |
| layer4-controls | not reproduced (outside input) |
| fig2c-cv-lower | reproduced |
| cohort-faster-firing | reproduced |
| cv-rate-matched | partially reproduced |
| cv-unmatched-reverses | partially reproduced |
| fig2d-cv-correlations | partially reproduced |
| cv-correlation-block-excess | partially reproduced |
| fig2f-pre-post-cv | not reproduced |
| fig3a-archetypes | reproduced (assumption) |
| fig3b-mixtures | not reproduced |
| fig3c-area-overlap | not reproduced |
| fig3d-anticipatory-gradient | reproduced (assumption) |
| anticipatory-direction-overlap | partially reproduced (assumption) |
| hits-vs-false-alarms | not reproduced |
| hits-vs-lick-bouts | partially reproduced (assumption) |

**How the analysis differs from the paper's.**

- Sessions and units: the paper does not say how its 103 sessions and 54,039 units were chosen. The
  104 sessions of the Sst-Cre and Vip-Cre mice, with units of quality "good" and presence ratio >=
  0.95 in the six areas, give 104 sessions, 54 mice and 54,045 units, and are used throughout (an
  assumption). The paper's 103rd-session exclusion could not be identified.
- The ISI-shuffle null of the split-half test uses 100 permutations per unit, not 1,000: 1,000 for all
  54,045 units would take about nine CPU hours, over the notebook's limit. On 300 units of three sessions
  the two give 225 and 229 units with p <= 0.05. The null rebuilds the train over the active block
  (reading 1a of the criterion); the whole-recording reading (1b) takes over an hour for all sessions
  and was run on 12 of them, where 8 of the 1,272 cohort units differ between the readings (net 4)
  and the discordant count is the same.
- The Wilcoxon signed-rank tests of responsiveness use the normal approximation with tie correction
  for every trial count.
- The seed sensitivity of the ISI null and of the latency jitter (which need the spike times again) was
  not computed under the four further seeds; the jitter's twenty realizations span 98.97 to 99.23%.
- The two-component mixture on durations taken from the 30 kHz mean waveform does not separate narrow
  and broad units; the claims on waveform classes are judged on the stored `waveform_duration`, the
  other reading the criterion lists.
- Layers come from the CCFv3 2017 25 um annotation (an outside input; the paper names no release), as
  BrainGlobe packages it or, when GIN cannot be reached, from the Allen Institute's download. In this
  run both were loaded; their volumes and structure lists are identical and every unit gets the same
  area, layer and boundary distance from each. A layer is assigned in any laminar CCF structure, as the criterion says; a rule that
  keeps only visual areas fits the paper's coverage, but is not in its Methods.
- The CV(t) of RW- is the mean of 20 count-matched draws (fig2c reading a), and claims
  fig2d-cv-correlations and cv-correlation-block-excess correlate that averaged series (an assumption).
  With a single draw, or with the correlations formed on each draw and then averaged over the 20, the
  RW- block's correlations fall below the RW+ block's (0.300 and 0.304 against 0.354), so the RW- excess
  depends on the averaging.

**What was found about the dataset.**

- The files hold 153 sessions from 81 mice; the paper's counts fit the 104 sessions of the Sst-Cre and
  Vip-Cre mice, within one session; the files give no reason to leave any of them out.
- The stream clocks agree: water follows the nearest lick by a median of 9.78 ms, and the units'
  summed response is locked to the change times of the stimulus table in every session.
- Blocks 0 and 5 are as the paper describes in every session.
- 1,508 of the 54,045 analysed units have a CCF coordinate of 0 or below, which gives them no layer.
- The units table's `waveform_duration` and the duration measured on the stored mean waveform agree
  at r = 0.66; the stored mean waveform is sampled at 30 kHz, so its durations take 60 values.
  Upsampled to 200 points, 97.5% agree within 0.02 ms, but 1,263 units have a duration to the following
  maximum more than 0.1 ms from the stored one (median 1.421 against 0.371 ms), and the files do not say
  how `waveform_duration` was measured for them.
